In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

BASE_DIR = Path(r"D:\khoane\MAHE\Project\dataset\UniPro")

metadata_file = BASE_DIR / "uniprot_human_reviewed_UP000005640_metadata.tsv"
fasta_file = BASE_DIR / "uniprot_human_reviewed_UP000005640_canonical.fasta"

df_meta = pd.read_csv(metadata_file, sep="\t")

print("Metadata shape:", df_meta.shape)
print()
print("Columns:")
print(df_meta.columns.tolist())

df_meta.head()

Metadata shape: (20416, 9)

Columns:
['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names', 'Organism', 'Length', 'Gene Names (primary)', 'Ensembl']


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Gene Names (primary),Ensembl
0,A0A087X1C5,reviewed,CP2D7_HUMAN,Cytochrome P450 2D7 (EC 1.14.14.1),CYP2D7,Homo sapiens (Human),515,CYP2D7,NaN
1,A0A096LP01,reviewed,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,Homo sapiens (Human),95,SMIM26,ENST00000411646.2 [A0A096LP01-1];ENST000004358...
2,A0A0B4J2F0,reviewed,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,PIGBOS1,ENST00000436697.3;ENST00000567948.1;
3,A0A0C5B5G6,reviewed,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,MT-RNR1,NaN
4,A0A0K2S4Q6,reviewed,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,Homo sapiens (Human),201,CD300H,ENST00000641031.1 [A0A0K2S4Q6-2];ENST000006417...


In [4]:
df_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20416 entries, 0 to 20415
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Entry                 20416 non-null  object
 1   Reviewed              20416 non-null  object
 2   Entry Name            20416 non-null  object
 3   Protein names         20416 non-null  object
 4   Gene Names            20304 non-null  object
 5   Organism              20416 non-null  object
 6   Length                20416 non-null  int64 
 7   Gene Names (primary)  20281 non-null  object
 8   Ensembl               19351 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.4+ MB


In [5]:
summary_meta = {
    "n_rows": len(df_meta),
    "n_unique_entries": df_meta["Entry"].nunique(),
    "n_unique_entry_names": df_meta["Entry Name"].nunique() if "Entry Name" in df_meta.columns else None,
    "n_unique_primary_gene_names": df_meta["Gene Names (primary)"].nunique() if "Gene Names (primary)" in df_meta.columns else None,
}

summary_meta

{'n_rows': 20416,
 'n_unique_entries': 20416,
 'n_unique_entry_names': 20416,
 'n_unique_primary_gene_names': 20190}

In [6]:
important_cols = [
    "Entry",
    "Entry Name",
    "Reviewed",
    "Protein names",
    "Gene Names",
    "Gene Names (primary)",
    "Organism",
    "Length",
    "Ensembl"
]

important_cols = [c for c in important_cols if c in df_meta.columns]

missing_meta = pd.DataFrame({
    "column": important_cols,
    "missing_count": [df_meta[c].isna().sum() for c in important_cols],
    "missing_pct": [df_meta[c].isna().mean() * 100 for c in important_cols]
})

missing_meta

,column,missing_count,missing_pct
0,Entry,0,0.000000
1,Entry Name,0,0.000000
2,Reviewed,0,0.000000
3,Protein names,0,0.000000
4,Gene Names,112,0.548589
5,Gene Names (primary),135,0.661246
6,Organism,0,0.000000
7,Length,0,0.000000
8,Ensembl,1065,5.216497


In [7]:
def parse_uniprot_fasta(fasta_path):
    records = []
    current_header = None
    current_seq = []

    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if current_header is not None:
                    records.append({
                        "fasta_header": current_header,
                        "sequence": "".join(current_seq)
                    })
                current_header = line[1:]
                current_seq = []
            else:
                current_seq.append(line)

        if current_header is not None:
            records.append({
                "fasta_header": current_header,
                "sequence": "".join(current_seq)
            })

    df = pd.DataFrame(records)

    # UniProt FASTA header thường dạng:
    # sp|ACCESSION|ENTRY_NAME Protein name OS=... OX=... GN=... PE=... SV=...
    df["Entry_from_fasta"] = df["fasta_header"].str.extract(r"^[^|]+\|([^|]+)\|")
    df["Entry_Name_from_fasta"] = df["fasta_header"].str.extract(r"^[^|]+\|[^|]+\|(\S+)")
    df["sequence_length_fasta"] = df["sequence"].str.len()

    return df

df_fasta = parse_uniprot_fasta(fasta_file)

print("FASTA shape:", df_fasta.shape)
df_fasta.head()

FASTA shape: (20416, 5)


,fasta_header,sequence,Entry_from_fasta,Entry_Name_from_fasta,sequence_length_fasta
0,sp|A0A087X1C5|CP2D7_HUMAN Cytochrome P450 2D7 ...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,A0A087X1C5,CP2D7_HUMAN,515
1,sp|A0A096LP01|SIM26_HUMAN Small integral membr...,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,A0A096LP01,SIM26_HUMAN,95
2,sp|A0A0B4J2F0|PIOS1_HUMAN Protein PIGBOS1 OS=H...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,A0A0B4J2F0,PIOS1_HUMAN,54
3,sp|A0A0C5B5G6|MOTSC_HUMAN Mitochondrial-derive...,MRWQEMGYIFYPRKLR,A0A0C5B5G6,MOTSC_HUMAN,16
4,sp|A0A0K2S4Q6|CD3CH_HUMAN Protein CD300H OS=Ho...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,A0A0K2S4Q6,CD3CH_HUMAN,201


In [8]:
summary_fasta = {
    "n_fasta_records": len(df_fasta),
    "n_unique_fasta_entries": df_fasta["Entry_from_fasta"].nunique(),
    "n_missing_accession_parsed": df_fasta["Entry_from_fasta"].isna().sum(),
    "min_sequence_length": df_fasta["sequence_length_fasta"].min(),
    "median_sequence_length": df_fasta["sequence_length_fasta"].median(),
    "max_sequence_length": df_fasta["sequence_length_fasta"].max(),
}

summary_fasta

{'n_fasta_records': 20416,
 'n_unique_fasta_entries': 20416,
 'n_missing_accession_parsed': 0,
 'min_sequence_length': 2,
 'median_sequence_length': 415.0,
 'max_sequence_length': 34350}

In [9]:
df_fasta["Entry_from_fasta"].duplicated().sum()

0

In [10]:
meta_entries = set(df_meta["Entry"].dropna().astype(str))
fasta_entries = set(df_fasta["Entry_from_fasta"].dropna().astype(str))

entry_match_summary = {
    "metadata_entries": len(meta_entries),
    "fasta_entries": len(fasta_entries),
    "entries_in_both": len(meta_entries & fasta_entries),
    "metadata_not_in_fasta": len(meta_entries - fasta_entries),
    "fasta_not_in_metadata": len(fasta_entries - meta_entries),
}

entry_match_summary

{'metadata_entries': 20416,
 'fasta_entries': 20416,
 'entries_in_both': 20416,
 'metadata_not_in_fasta': 0,
 'fasta_not_in_metadata': 0}

In [11]:
df_protein_universe = df_meta.merge(
    df_fasta[
        [
            "Entry_from_fasta",
            "fasta_header",
            "sequence",
            "sequence_length_fasta"
        ]
    ],
    left_on="Entry",
    right_on="Entry_from_fasta",
    how="left"
)

print("Protein universe shape:", df_protein_universe.shape)
df_protein_universe.head()

Protein universe shape: (20416, 13)


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Gene Names (primary),Ensembl,Entry_from_fasta,fasta_header,sequence,sequence_length_fasta
0,A0A087X1C5,reviewed,CP2D7_HUMAN,Cytochrome P450 2D7 (EC 1.14.14.1),CYP2D7,Homo sapiens (Human),515,CYP2D7,NaN,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN Cytochrome P450 2D7 ...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,515
1,A0A096LP01,reviewed,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,Homo sapiens (Human),95,SMIM26,ENST00000411646.2 [A0A096LP01-1];ENST000004358...,A0A096LP01,sp|A0A096LP01|SIM26_HUMAN Small integral membr...,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95
2,A0A0B4J2F0,reviewed,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,PIGBOS1,ENST00000436697.3;ENST00000567948.1;,A0A0B4J2F0,sp|A0A0B4J2F0|PIOS1_HUMAN Protein PIGBOS1 OS=H...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54
3,A0A0C5B5G6,reviewed,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,MT-RNR1,NaN,A0A0C5B5G6,sp|A0A0C5B5G6|MOTSC_HUMAN Mitochondrial-derive...,MRWQEMGYIFYPRKLR,16
4,A0A0K2S4Q6,reviewed,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,Homo sapiens (Human),201,CD300H,ENST00000641031.1 [A0A0K2S4Q6-2];ENST000006417...,A0A0K2S4Q6,sp|A0A0K2S4Q6|CD3CH_HUMAN Protein CD300H OS=Ho...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201


In [12]:
df_meta["Ensembl"].dropna().head(20).tolist()

['ENST00000411646.2 [A0A096LP01-1];ENST00000435844.3 [A0A096LP01-2];',
 'ENST00000436697.3;ENST00000567948.1;',
 'ENST00000641031.1 [A0A0K2S4Q6-2];ENST00000641710.1 [A0A0K2S4Q6-1];ENST00000709217.2 [A0A0K2S4Q6-1];ENST00000709218.1 [A0A0K2S4Q6-2];',
 'ENST00000374922.9;ENST00000423617.2;ENST00000637096.1;',
 'ENST00000637218.2 [A0A1B0GTW7-1];ENST00000644000.1 [A0A1B0GTW7-2];',
 'ENST00000646160.1;',
 'ENST00000484897.4;',
 'ENST00000393469.8 [A0AV02-1];ENST00000430155.6 [A0AV02-3];ENST00000469902.6 [A0AV02-1];',
 'ENST00000295971.12 [A0AV96-1];ENST00000381793.6 [A0AV96-1];ENST00000381795.10 [A0AV96-2];',
 'ENST00000343187.8 [A0AVF1-3];ENST00000430935.5 [A0AVF1-2];ENST00000464848.5 [A0AVF1-1];',
 'ENST00000303277.6 [A0AVI4-2];ENST00000382936.8 [A0AVI4-1];',
 'ENST00000250024.9;ENST00000527884.5;ENST00000620009.4;',
 'ENST00000322244.10 [A0AVT1-1];ENST00000420827.2 [A0AVT1-3];',
 'ENST00000652148.1 [A0FGR8-2];',
 'ENST00000389567.9 [A0FGR9-1];ENST00000490835.5 [A0FGR9-2];',
 'ENST00000337

In [13]:
df_meta["has_ENSG_in_Ensembl"] = (
    df_meta["Ensembl"]
    .fillna("")
    .astype(str)
    .str.contains(r"ENSG\d+", regex=True)
)

df_meta["has_ENSG_in_Ensembl"].value_counts()

has_ENSG_in_Ensembl
False    20416
Name: count, dtype: int64

In [14]:
df_meta["has_ENSG_in_Ensembl"].mean() * 100

0.0

In [15]:
import pandas as pd
import re

def extract_enst_ids(x):
    if pd.isna(x):
        return []
    x = str(x)
    # Lấy ENST, bỏ version phía sau nếu có
    ids = re.findall(r"ENST\d+", x)
    return sorted(set(ids))

df_protein_universe["ensembl_transcript_ids"] = (
    df_protein_universe["Ensembl"].apply(extract_enst_ids)
)

df_protein_universe["n_ensembl_transcripts"] = (
    df_protein_universe["ensembl_transcript_ids"].apply(len)
)

df_protein_universe["has_enst"] = (
    df_protein_universe["n_ensembl_transcripts"] > 0
)

df_protein_universe["has_enst"].value_counts()

has_enst
True     19351
False     1065
Name: count, dtype: int64

In [16]:
uniprot_enst_map = df_protein_universe[
    [
        "Entry",
        "Entry Name",
        "Protein names",
        "Gene Names",
        "Gene Names (primary)",
        "Length",
        "sequence",
        "sequence_length_fasta",
        "ensembl_transcript_ids"
    ]
].copy()

uniprot_enst_map = (
    uniprot_enst_map
    .explode("ensembl_transcript_ids")
    .rename(columns={
        "Entry": "uniprot_accession",
        "Entry Name": "uniprot_entry_name",
        "Gene Names (primary)": "primary_gene_name",
        "ensembl_transcript_ids": "ensembl_transcript_id"
    })
)

uniprot_enst_map = uniprot_enst_map.dropna(
    subset=["ensembl_transcript_id"]
).copy()

print("UniProt-ENST mapping shape:", uniprot_enst_map.shape)
uniprot_enst_map.head()

UniProt-ENST mapping shape: (50864, 9)


,uniprot_accession,uniprot_entry_name,Protein names,Gene Names,primary_gene_name,Length,sequence,sequence_length_fasta,ensembl_transcript_id
1,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,SMIM26,95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,ENST00000411646
1,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,SMIM26,95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,ENST00000435844
2,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,PIGBOS1,54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,ENST00000436697
2,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,PIGBOS1,54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,ENST00000567948
4,A0A0K2S4Q6,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,CD300H,201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201,ENST00000641031


In [17]:
summary_uniprot_enst = {
    "n_uniprot_entries_with_enst": uniprot_enst_map["uniprot_accession"].nunique(),
    "n_unique_enst": uniprot_enst_map["ensembl_transcript_id"].nunique(),
    "n_rows_uniprot_enst": len(uniprot_enst_map),
}

summary_uniprot_enst

{'n_uniprot_entries_with_enst': 19351,
 'n_unique_enst': 50864,
 'n_rows_uniprot_enst': 50864}

In [18]:
duplicate_primary_genes = (
    df_protein_universe["Gene Names (primary)"]
    .value_counts()
    .reset_index()
)

duplicate_primary_genes.columns = ["primary_gene_name", "n_entries"]

duplicate_primary_genes = duplicate_primary_genes[
    duplicate_primary_genes["n_entries"] > 1
]

print("Number of duplicated primary gene symbols:", len(duplicate_primary_genes))
duplicate_primary_genes.head(30)

Number of duplicated primary gene symbols: 56


,primary_gene_name,n_entries
0,MT-RNR2,7
1,ERVK-6,5
2,ERVK-7,5
3,HERVK_113,5
4,ERVK-8,5
5,ERVK-19,5
6,ERVK-9,5
7,GNAS,4
8,ERVK-21,4
9,ERVK-24,4


In [21]:
df_biomart = pd.read_csv(
    biomart_file,
    sep="\t",
    dtype={"Chromosome/scaffold name": str},
    low_memory=False
)

print("BioMart shape:", df_biomart.shape)
print(df_biomart.columns.tolist())
df_biomart.head()

BioMart shape: (533740, 11)
['Gene stable ID', 'Gene stable ID version', 'Transcript stable ID', 'Transcript stable ID version', 'Gene name', 'Gene type', 'Transcript type', 'Chromosome/scaffold name', 'Gene start (bp)', 'Gene end (bp)', 'Strand']


,Gene stable ID,Gene stable ID version,Transcript stable ID,Transcript stable ID version,Gene name,Gene type,Transcript type,Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand
0,ENSG00000210049,ENSG00000210049.1,ENST00000387314,ENST00000387314.1,MT-TF,Mt_tRNA,Mt_tRNA,MT,577,647,1
1,ENSG00000211459,ENSG00000211459.2,ENST00000389680,ENST00000389680.2,MT-RNR1,Mt_rRNA,Mt_rRNA,MT,648,1601,1
2,ENSG00000210077,ENSG00000210077.1,ENST00000387342,ENST00000387342.1,MT-TV,Mt_tRNA,Mt_tRNA,MT,1602,1670,1
3,ENSG00000210082,ENSG00000210082.2,ENST00000387347,ENST00000387347.2,MT-RNR2,Mt_rRNA,Mt_rRNA,MT,1671,3229,1
4,ENSG00000209082,ENSG00000209082.1,ENST00000386347,ENST00000386347.1,MT-TL1,Mt_tRNA,Mt_tRNA,MT,3230,3304,1


In [22]:
df_biomart_clean = df_biomart.rename(columns={
    "Gene stable ID": "ensembl_gene_id",
    "Gene stable ID version": "ensembl_gene_id_version",
    "Transcript stable ID": "ensembl_transcript_id",
    "Transcript stable ID version": "ensembl_transcript_id_version",
    "Gene name": "ensembl_gene_name",
    "Gene type": "gene_type",
    "Transcript type": "transcript_type",
    "Chromosome/scaffold name": "chromosome_or_scaffold",
    "Gene start (bp)": "gene_start_bp",
    "Gene end (bp)": "gene_end_bp",
    "Strand": "strand"
}).copy()

df_biomart_clean.head()

,ensembl_gene_id,ensembl_gene_id_version,ensembl_transcript_id,ensembl_transcript_id_version,ensembl_gene_name,gene_type,transcript_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,ENSG00000210049,ENSG00000210049.1,ENST00000387314,ENST00000387314.1,MT-TF,Mt_tRNA,Mt_tRNA,MT,577,647,1
1,ENSG00000211459,ENSG00000211459.2,ENST00000389680,ENST00000389680.2,MT-RNR1,Mt_rRNA,Mt_rRNA,MT,648,1601,1
2,ENSG00000210077,ENSG00000210077.1,ENST00000387342,ENST00000387342.1,MT-TV,Mt_tRNA,Mt_tRNA,MT,1602,1670,1
3,ENSG00000210082,ENSG00000210082.2,ENST00000387347,ENST00000387347.2,MT-RNR2,Mt_rRNA,Mt_rRNA,MT,1671,3229,1
4,ENSG00000209082,ENSG00000209082.1,ENST00000386347,ENST00000386347.1,MT-TL1,Mt_tRNA,Mt_tRNA,MT,3230,3304,1


In [23]:
biomart_summary = {
    "n_rows": len(df_biomart_clean),
    "n_unique_genes_ENSG": df_biomart_clean["ensembl_gene_id"].nunique(),
    "n_unique_transcripts_ENST": df_biomart_clean["ensembl_transcript_id"].nunique(),
    "n_missing_ENSG": df_biomart_clean["ensembl_gene_id"].isna().sum(),
    "n_missing_ENST": df_biomart_clean["ensembl_transcript_id"].isna().sum(),
}

biomart_summary

{'n_rows': 533740,
 'n_unique_genes_ENSG': 86369,
 'n_unique_transcripts_ENST': 533740,
 'n_missing_ENSG': 0,
 'n_missing_ENST': 0}

In [24]:
enst_to_gene_check = (
    df_biomart_clean
    .groupby("ensembl_transcript_id")["ensembl_gene_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

enst_to_gene_check

ensembl_gene_id
1    533740
Name: count, dtype: int64

In [25]:
enst_to_ensg_map = (
    df_biomart_clean[
        [
            "ensembl_transcript_id",
            "ensembl_gene_id",
            "ensembl_gene_name",
            "gene_type",
            "transcript_type",
            "chromosome_or_scaffold",
            "gene_start_bp",
            "gene_end_bp",
            "strand"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("ENST → ENSG mapping shape:", enst_to_ensg_map.shape)
enst_to_ensg_map.head()

ENST → ENSG mapping shape: (533740, 9)


,ensembl_transcript_id,ensembl_gene_id,ensembl_gene_name,gene_type,transcript_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,ENST00000387314,ENSG00000210049,MT-TF,Mt_tRNA,Mt_tRNA,MT,577,647,1
1,ENST00000389680,ENSG00000211459,MT-RNR1,Mt_rRNA,Mt_rRNA,MT,648,1601,1
2,ENST00000387342,ENSG00000210077,MT-TV,Mt_tRNA,Mt_tRNA,MT,1602,1670,1
3,ENST00000387347,ENSG00000210082,MT-RNR2,Mt_rRNA,Mt_rRNA,MT,1671,3229,1
4,ENST00000386347,ENSG00000209082,MT-TL1,Mt_tRNA,Mt_tRNA,MT,3230,3304,1


In [26]:
uniprot_enst_ensg_map = uniprot_enst_map.merge(
    enst_to_ensg_map,
    on="ensembl_transcript_id",
    how="left"
)

print("UniProt ENST → ENSG merged shape:", uniprot_enst_ensg_map.shape)
uniprot_enst_ensg_map.head()

UniProt ENST → ENSG merged shape: (50864, 17)


,uniprot_accession,uniprot_entry_name,Protein names,Gene Names,primary_gene_name,Length,sequence,sequence_length_fasta,ensembl_transcript_id,ensembl_gene_id,ensembl_gene_name,gene_type,transcript_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,SMIM26,95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,ENST00000411646,ENSG00000232388,SMIM26,protein_coding,protein_coding,20,18567347,18569563,1
1,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,SMIM26,95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,ENST00000435844,ENSG00000232388,SMIM26,protein_coding,protein_coding,20,18567347,18569563,1
2,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,PIGBOS1,54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,ENST00000436697,ENSG00000225973,PIGBOS1,protein_coding,protein_coding,15,55317175,55319419,-1
3,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,PIGBOS1,54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,ENST00000567948,ENSG00000225973,PIGBOS1,protein_coding,protein_coding,15,55317175,55319419,-1
4,A0A0K2S4Q6,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,CD300H,201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201,ENST00000641031,ENSG00000284690,CD300H,protein_coding,protein_coding_LoF,17,74560701,74567343,-1


In [27]:
enst_mapping_summary = {
    "n_uniprot_enst_rows": len(uniprot_enst_ensg_map),
    "n_rows_with_ENSG": uniprot_enst_ensg_map["ensembl_gene_id"].notna().sum(),
    "n_rows_without_ENSG": uniprot_enst_ensg_map["ensembl_gene_id"].isna().sum(),
    "pct_rows_with_ENSG": round(
        uniprot_enst_ensg_map["ensembl_gene_id"].notna().mean() * 100,
        4
    ),
}

enst_mapping_summary

{'n_uniprot_enst_rows': 50864,
 'n_rows_with_ENSG': 50864,
 'n_rows_without_ENSG': 0,
 'pct_rows_with_ENSG': 100.0}

In [28]:
uniprot_mapping_status = (
    uniprot_enst_ensg_map
    .groupby("uniprot_accession")["ensembl_gene_id"]
    .apply(lambda s: s.notna().any())
    .reset_index(name="has_ENSG_mapping")
)

uniprot_mapping_status["has_ENSG_mapping"].value_counts()

has_ENSG_mapping
True    19351
Name: count, dtype: int64

In [29]:
uniprot_mapping_coverage = {
    "n_uniprot_entries_with_ENST": uniprot_enst_map["uniprot_accession"].nunique(),
    "n_uniprot_entries_mapped_to_ENSG": int(
        uniprot_mapping_status["has_ENSG_mapping"].sum()
    ),
    "n_uniprot_entries_without_ENSG_after_BioMart": int(
        (~uniprot_mapping_status["has_ENSG_mapping"]).sum()
    ),
    "pct_uniprot_entries_mapped_to_ENSG": round(
        uniprot_mapping_status["has_ENSG_mapping"].mean() * 100,
        4
    )
}

uniprot_mapping_coverage

{'n_uniprot_entries_with_ENST': 19351,
 'n_uniprot_entries_mapped_to_ENSG': 19351,
 'n_uniprot_entries_without_ENSG_after_BioMart': 0,
 'pct_uniprot_entries_mapped_to_ENSG': 100.0}

In [30]:
uniprot_to_ensg_counts = (
    uniprot_enst_ensg_map
    .dropna(subset=["ensembl_gene_id"])
    .groupby("uniprot_accession")["ensembl_gene_id"]
    .nunique()
    .reset_index(name="n_unique_ENSG")
)

uniprot_to_ensg_counts["n_unique_ENSG"].value_counts().sort_index()

n_unique_ENSG
1     17999
2      1052
3        80
4        43
5        40
6        29
7        50
8        32
9         3
10       14
12        2
13        1
14        2
17        1
21        1
22        1
24        1
Name: count, dtype: int64

In [31]:
multi_ensg_uniprot = uniprot_to_ensg_counts[
    uniprot_to_ensg_counts["n_unique_ENSG"] > 1
]

print("Number of UniProt entries mapping to >1 ENSG:", len(multi_ensg_uniprot))
multi_ensg_uniprot.head(20)

Number of UniProt entries mapping to >1 ENSG: 1352


,uniprot_accession,n_unique_ENSG
2,A0A075B6H7,2
12,A0A075B6J1,2
13,A0A075B6J2,2
27,A0A075B6P5,2
28,A0A075B6Q5,2
30,A0A075B6R2,2
35,A0A075B6S5,2
37,A0A075B6S9,2
63,A0A087WV62,2
69,A0A087WXS9,3


In [32]:
category_a_file = r"D:\khoane\MAHE\Project\dataset\GWAS\t2d_positive_labels_v1_modeling_minimal.csv"

df_pos = pd.read_csv(category_a_file)

print("Category A positive shape:", df_pos.shape)
print(df_pos.columns.tolist())
df_pos.head()

Category A positive shape: (911, 12)
['target_id', 'target_symbol', 'target_name', 'association_score', 'positive_tier', 'label', 'label_name', 'label_source', 'label_rule', 'label_version', 'disease_id', 'disease_name']


,target_id,target_symbol,target_name,association_score,positive_tier,label,label_name,label_source,label_rule,label_version,disease_id,disease_name
0,ENSG00000187486,KCNJ11,potassium inwardly rectifying channel subfamil...,0.865142,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,MONDO_0005148,type 2 diabetes mellitus
1,ENSG00000006071,ABCC8,ATP binding cassette subfamily C member 8,0.864850,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,MONDO_0005148,type 2 diabetes mellitus
2,ENSG00000106633,GCK,glucokinase,0.861234,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,MONDO_0005148,type 2 diabetes mellitus
3,ENSG00000132170,PPARG,peroxisome proliferator activated receptor gamma,0.848609,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,MONDO_0005148,type 2 diabetes mellitus
4,ENSG00000171105,INSR,insulin receptor,0.788737,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,MONDO_0005148,type 2 diabetes mellitus


In [33]:
protein_ensg_map_raw = (
    uniprot_enst_ensg_map
    .dropna(subset=["ensembl_gene_id"])
    [
        [
            "uniprot_accession",
            "uniprot_entry_name",
            "Protein names",
            "Gene Names",
            "primary_gene_name",
            "Length",
            "sequence",
            "sequence_length_fasta",
            "ensembl_gene_id",
            "ensembl_gene_name",
            "gene_type",
            "chromosome_or_scaffold",
            "gene_start_bp",
            "gene_end_bp",
            "strand"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("Protein-ENSG raw shape:", protein_ensg_map_raw.shape)
protein_ensg_map_raw.head()

Protein-ENSG raw shape: (21811, 15)


,uniprot_accession,uniprot_entry_name,Protein names,Gene Names,primary_gene_name,Length,sequence,sequence_length_fasta,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,SMIM26,95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,ENSG00000232388,SMIM26,protein_coding,20,18567347,18569563,1
2,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,PIGBOS1,54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,ENSG00000225973,PIGBOS1,protein_coding,15,55317175,55319419,-1
4,A0A0K2S4Q6,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,CD300H,201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201,ENSG00000284690,CD300H,protein_coding,17,74560701,74567343,-1
6,A0A0K2S4Q6,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,CD300H,201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,201,ENSG00000291925,CD300H,protein_coding,HG2580_PATCH,136619,143957,-1
8,A0A0U1RRE5,NBDY_HUMAN,Negative regulator of P-body association (P-bo...,NBDY LINC01420,NBDY,68,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,68,ENSG00000204272,NBDY,protein_coding,X,56729219,56819179,1


In [34]:
uniprot_to_ensg_counts = (
    protein_ensg_map_raw
    .groupby("uniprot_accession")["ensembl_gene_id"]
    .nunique()
    .reset_index(name="n_unique_ENSG_per_uniprot")
)

protein_ensg_map_raw = protein_ensg_map_raw.merge(
    uniprot_to_ensg_counts,
    on="uniprot_accession",
    how="left"
)

protein_ensg_map_raw["uniprot_to_ensg_status"] = np.where(
    protein_ensg_map_raw["n_unique_ENSG_per_uniprot"] == 1,
    "one_uniprot_to_one_ensg",
    "one_uniprot_to_multiple_ensg"
)

protein_ensg_map_raw["uniprot_to_ensg_status"].value_counts()

uniprot_to_ensg_status
one_uniprot_to_one_ensg         17999
one_uniprot_to_multiple_ensg     3812
Name: count, dtype: int64

In [35]:
df_pos_protein_map_raw = df_pos.merge(
    protein_ensg_map_raw,
    left_on="target_id",
    right_on="ensembl_gene_id",
    how="left"
)

print("Positive × protein raw mapping shape:", df_pos_protein_map_raw.shape)
df_pos_protein_map_raw.head()

Positive × protein raw mapping shape: (912, 29)


,target_id,target_symbol,target_name,association_score,positive_tier,label,label_name,label_source,label_rule,label_version,...,sequence_length_fasta,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,n_unique_ENSG_per_uniprot,uniprot_to_ensg_status
0,ENSG00000187486,KCNJ11,potassium inwardly rectifying channel subfamil...,0.865142,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,...,390.0,ENSG00000187486,KCNJ11,protein_coding,11,17365172.0,17389353.0,-1.0,1.0,one_uniprot_to_one_ensg
1,ENSG00000006071,ABCC8,ATP binding cassette subfamily C member 8,0.864850,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,...,1581.0,ENSG00000006071,ABCC8,protein_coding,11,17392498.0,17476894.0,-1.0,1.0,one_uniprot_to_one_ensg
2,ENSG00000106633,GCK,glucokinase,0.861234,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,...,465.0,ENSG00000106633,GCK,protein_coding,7,44143213.0,44198170.0,-1.0,1.0,one_uniprot_to_one_ensg
3,ENSG00000132170,PPARG,peroxisome proliferator activated receptor gamma,0.848609,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,...,505.0,ENSG00000132170,PPARG,protein_coding,3,12287368.0,12434574.0,1.0,1.0,one_uniprot_to_one_ensg
4,ENSG00000171105,INSR,insulin receptor,0.788737,Tier_A_high_confidence_ge_0.50,1,T2D_positive,Open Targets Platform,OpenTargets_association_score_ge_0.30,Positive_Label_v1,...,1382.0,ENSG00000171105,INSR,protein_coding,19,7112255.0,7294414.0,-1.0,1.0,one_uniprot_to_one_ensg


In [36]:
positive_mapping_summary = {
    "n_category_a_positive_rows": len(df_pos),
    "n_unique_category_a_targets": df_pos["target_id"].nunique(),
    "n_positive_targets_with_protein_mapping": df_pos_protein_map_raw.loc[
        df_pos_protein_map_raw["uniprot_accession"].notna(),
        "target_id"
    ].nunique(),
    "n_positive_targets_without_protein_mapping": df_pos["target_id"].nunique()
    - df_pos_protein_map_raw.loc[
        df_pos_protein_map_raw["uniprot_accession"].notna(),
        "target_id"
    ].nunique(),
}

positive_mapping_summary["pct_positive_targets_with_protein_mapping"] = round(
    positive_mapping_summary["n_positive_targets_with_protein_mapping"]
    / positive_mapping_summary["n_unique_category_a_targets"]
    * 100,
    4
)

positive_mapping_summary

{'n_category_a_positive_rows': 911,
 'n_unique_category_a_targets': 911,
 'n_positive_targets_with_protein_mapping': 905,
 'n_positive_targets_without_protein_mapping': 6,
 'pct_positive_targets_with_protein_mapping': 99.3414}

In [37]:
positive_ambiguous_summary = (
    df_pos_protein_map_raw
    .dropna(subset=["uniprot_accession"])
    .groupby("target_id")
    .agg(
        target_symbol=("target_symbol", "first"),
        n_uniprot_accessions=("uniprot_accession", "nunique"),
        n_unique_sequences=("sequence", "nunique"),
        max_uniprot_to_ensg_count=("n_unique_ENSG_per_uniprot", "max")
    )
    .reset_index()
)

positive_ambiguous_summary["protein_mapping_status"] = np.select(
    [
        positive_ambiguous_summary["n_uniprot_accessions"] == 1,
        positive_ambiguous_summary["n_uniprot_accessions"] > 1
    ],
    [
        "one_positive_gene_to_one_uniprot",
        "one_positive_gene_to_multiple_uniprot"
    ],
    default="no_mapping"
)

positive_ambiguous_summary["protein_mapping_status"].value_counts()

protein_mapping_status
one_positive_gene_to_one_uniprot         904
one_positive_gene_to_multiple_uniprot      1
Name: count, dtype: int64

In [38]:
positive_multi_uniprot = positive_ambiguous_summary[
    positive_ambiguous_summary["n_uniprot_accessions"] > 1
].sort_values("n_uniprot_accessions", ascending=False)

positive_multi_uniprot.head(30)

,target_id,target_symbol,n_uniprot_accessions,n_unique_sequences,max_uniprot_to_ensg_count,protein_mapping_status
30,ENSG00000021645,NRXN3,2,2,1.0,one_positive_gene_to_multiple_uniprot


In [39]:
ambiguous_uniprot_set = set(
    uniprot_to_ensg_counts.loc[
        uniprot_to_ensg_counts["n_unique_ENSG_per_uniprot"] > 1,
        "uniprot_accession"
    ]
)

positive_using_ambiguous_uniprot = df_pos_protein_map_raw[
    df_pos_protein_map_raw["uniprot_accession"].isin(ambiguous_uniprot_set)
].copy()

print("Positive rows using ambiguous UniProt entries:", positive_using_ambiguous_uniprot.shape)

positive_using_ambiguous_uniprot[
    [
        "target_id",
        "target_symbol",
        "association_score",
        "uniprot_accession",
        "uniprot_entry_name",
        "primary_gene_name",
        "ensembl_gene_id",
        "ensembl_gene_name",
        "n_unique_ENSG_per_uniprot"
    ]
].drop_duplicates().head(50)

Positive rows using ambiguous UniProt entries: (28, 29)


,target_id,target_symbol,association_score,uniprot_accession,uniprot_entry_name,primary_gene_name,ensembl_gene_id,ensembl_gene_name,n_unique_ENSG_per_uniprot
5,ENSG00000275410,HNF1B,0.784586,P35680,HNF1B_HUMAN,HNF1B,ENSG00000275410,HNF1B,2.0
45,ENSG00000023228,NDUFS1,0.608933,P28331,NDUS1_HUMAN,NDUFS1,ENSG00000023228,NDUFS1,2.0
52,ENSG00000174886,NDUFA11,0.608009,Q86Y39,NDUAB_HUMAN,NDUFA11,ENSG00000174886,NDUFA11,2.0
53,ENSG00000184983,NDUFA6,0.608009,P56556,NDUA6_HUMAN,NDUFA6,ENSG00000184983,NDUFA6,5.0
55,ENSG00000140990,NDUFB10,0.608009,O96000,NDUBA_HUMAN,NDUFB10,ENSG00000140990,NDUFB10,2.0
66,ENSG00000170906,NDUFA3,0.607085,O95167,NDUA3_HUMAN,NDUFA3,ENSG00000170906,NDUFA3,10.0
79,ENSG00000171298,GAA,0.581838,P10253,LYAG_HUMAN,GAA,ENSG00000171298,GAA,2.0
84,ENSG00000257743,MGAM2,0.568445,Q2M2H8,MGAL_HUMAN,MGAM2,ENSG00000257743,MGAM2,2.0
145,ENSG00000139146,SINHCAF,0.519872,Q9NP50,SHCAF_HUMAN,SINHCAF,ENSG00000139146,SINHCAF,2.0
147,ENSG00000069696,DRD4,0.518031,P21917,DRD4_HUMAN,DRD4,ENSG00000069696,DRD4,2.0


In [40]:
unmapped_positive_targets = (
    df_pos_protein_map_raw[
        df_pos_protein_map_raw["uniprot_accession"].isna()
    ][
        [
            "target_id",
            "target_symbol",
            "target_name",
            "association_score",
            "positive_tier"
        ]
    ]
    .drop_duplicates()
    .sort_values("association_score", ascending=False)
)

unmapped_positive_targets

,target_id,target_symbol,target_name,association_score,positive_tier
114,ENSG00000166436,TRIM66,tripartite motif containing 66,0.535019,Tier_A_high_confidence_ge_0.50
135,ENSG00000175164,ABO,"ABO, alpha 1-3-N-acetylgalactosaminyltransfera...",0.527205,Tier_A_high_confidence_ge_0.50
229,ENSG00000084734,GCKR,glucokinase regulator,0.477355,Tier_B_strong_positive_0.30_to_0.50
278,ENSG00000174326,SLC16A11,solute carrier family 16 member 11,0.459762,Tier_B_strong_positive_0.30_to_0.50
339,ENSG00000060749,QSER1,glutamine and serine rich 1,0.438557,Tier_B_strong_positive_0.30_to_0.50
892,ENSG00000293584,ENSG00000293584,novel protein,0.304489,Tier_B_strong_positive_0.30_to_0.50


In [41]:
nrxn3_mapping = (
    df_pos_protein_map_raw[
        df_pos_protein_map_raw["target_symbol"] == "NRXN3"
    ][
        [
            "target_id",
            "target_symbol",
            "association_score",
            "uniprot_accession",
            "uniprot_entry_name",
            "Protein names",
            "primary_gene_name",
            "Length",
            "sequence_length_fasta",
            "ensembl_gene_id",
            "ensembl_gene_name",
            "n_unique_ENSG_per_uniprot"
        ]
    ]
    .drop_duplicates()
)

nrxn3_mapping

,target_id,target_symbol,association_score,uniprot_accession,uniprot_entry_name,Protein names,primary_gene_name,Length,sequence_length_fasta,ensembl_gene_id,ensembl_gene_name,n_unique_ENSG_per_uniprot
402,ENSG00000021645,NRXN3,0.416789,Q9HDB5,NRX3B_HUMAN,Neurexin-3-beta (Neurexin III-beta) [Cleaved i...,NRXN3,637.0,637.0,ENSG00000021645,NRXN3,1.0
403,ENSG00000021645,NRXN3,0.416789,Q9Y4C0,NRX3A_HUMAN,Neurexin-3 (Neurexin III-alpha) (Neurexin-3-al...,NRXN3,1643.0,1643.0,ENSG00000021645,NRXN3,1.0


In [42]:
ambiguous_positive_summary = {
    "n_rows": len(positive_using_ambiguous_uniprot),
    "n_unique_positive_targets": positive_using_ambiguous_uniprot["target_id"].nunique(),
    "n_unique_uniprot_accessions": positive_using_ambiguous_uniprot["uniprot_accession"].nunique(),
}

ambiguous_positive_summary

{'n_rows': 28,
 'n_unique_positive_targets': 28,
 'n_unique_uniprot_accessions': 28}

In [43]:
import numpy as np

df_positive_mapping_audit = df_pos_protein_map_raw.copy()

df_positive_mapping_audit["positive_protein_mapping_status"] = np.select(
    [
        df_positive_mapping_audit["uniprot_accession"].isna(),

        df_positive_mapping_audit["target_symbol"].eq("NRXN3"),

        df_positive_mapping_audit["n_unique_ENSG_per_uniprot"].fillna(0) > 1,

        df_positive_mapping_audit["uniprot_accession"].notna()
    ],
    [
        "unmapped_no_uniprot_sequence_yet",

        "mapped_target_has_multiple_uniprot_sequences",

        "mapped_exact_target_but_uniprot_crossmaps_multiple_ensg",

        "mapped_clean_one_target_one_uniprot"
    ],
    default="check_manually"
)

df_positive_mapping_audit["positive_protein_mapping_status"].value_counts()

positive_protein_mapping_status
mapped_clean_one_target_one_uniprot                        876
mapped_exact_target_but_uniprot_crossmaps_multiple_ensg     28
unmapped_no_uniprot_sequence_yet                             6
mapped_target_has_multiple_uniprot_sequences                 2
Name: count, dtype: int64

In [44]:
unmapped_symbols = unmapped_positive_targets["target_symbol"].tolist()

fallback_symbol_candidates = (
    df_protein_universe[
        df_protein_universe["Gene Names (primary)"].isin(unmapped_symbols)
    ][
        [
            "Entry",
            "Entry Name",
            "Protein names",
            "Gene Names",
            "Gene Names (primary)",
            "Length",
            "Ensembl",
            "sequence",
            "sequence_length_fasta"
        ]
    ]
    .copy()
    .sort_values(["Gene Names (primary)", "Length"], ascending=[True, False])
)

print("Fallback symbol candidate shape:", fallback_symbol_candidates.shape)
fallback_symbol_candidates

Fallback symbol candidate shape: (5, 9)


,Entry,Entry Name,Protein names,Gene Names,Gene Names (primary),Length,Ensembl,sequence,sequence_length_fasta
2897,P16442,BGAT_HUMAN,Histo-blood group ABO system transferase (Fuco...,ABO,ABO,354,NaN,MAEVLRTLAGKPKCHALRPMILFLIMLVLVLFGYGVLSPRSLMPGS...,354
6201,Q14397,GCKR_HUMAN,Glucokinase regulatory protein (GKRP) (Glucoki...,GCKR,GCKR,625,NaN,MPGTKRFQHVIETPEPGKWELSGYEAAVPITEKSNPLTQDLDKADA...,625
17296,Q2KHR3,QSER1_HUMAN,Glutamine and serine-rich protein 1,QSER1,QSER1,1735,NaN,MNFLSTAESRTAQAAASGTTLLPQFRAPSWQTGMHSSAATELFATG...,1735
9314,Q8NCK7,MOT11_HUMAN,Monocarboxylate transporter 11 (MCT 11) (Solut...,SLC16A11 MCT11,SLC16A11,447,NaN,MTPQPAGPPDGGWGWVVAAAAFAINGLSYGLLRSLGLAFPDLAEHF...,447
14817,O15016,TRI66_HUMAN,Tripartite motif-containing protein 66,TRIM66 C11orf29 KIAA0298,TRIM66,1351,NaN,MSFMGLPLAGQKHCPKSGQMEAMVMTCSLCHQDLPGMGSHLLSCQH...,1351


In [45]:
fallback_symbol_summary = (
    fallback_symbol_candidates
    .groupby("Gene Names (primary)")
    .agg(
        n_uniprot_entries=("Entry", "nunique"),
        uniprot_accessions=("Entry", lambda x: sorted(set(x))),
        entry_names=("Entry Name", lambda x: sorted(set(x))),
        lengths=("Length", lambda x: sorted(set(x), reverse=True))
    )
    .reset_index()
    .rename(columns={"Gene Names (primary)": "target_symbol"})
)

fallback_symbol_summary

,target_symbol,n_uniprot_entries,uniprot_accessions,entry_names,lengths
0,ABO,1,[P16442],[BGAT_HUMAN],[354]
1,GCKR,1,[Q14397],[GCKR_HUMAN],[625]
2,QSER1,1,[Q2KHR3],[QSER1_HUMAN],[1735]
3,SLC16A11,1,[Q8NCK7],[MOT11_HUMAN],[447]
4,TRIM66,1,[O15016],[TRI66_HUMAN],[1351]


In [46]:
from pathlib import Path
import pandas as pd
import numpy as np

output_dir = BASE_DIR / "category_b_outputs"
output_dir.mkdir(exist_ok=True)

In [47]:
direct_mapped_positive = df_pos_protein_map_raw[
    df_pos_protein_map_raw["uniprot_accession"].notna()
].copy()

print("Direct mapped positive rows:", direct_mapped_positive.shape)
print("Direct mapped unique targets:", direct_mapped_positive["target_id"].nunique())

Direct mapped positive rows: (906, 29)
Direct mapped unique targets: 905


In [48]:
biomart_gene_info = (
    df_biomart_clean[
        [
            "ensembl_gene_id",
            "ensembl_gene_name",
            "gene_type",
            "chromosome_or_scaffold",
            "gene_start_bp",
            "gene_end_bp",
            "strand"
        ]
    ]
    .drop_duplicates(subset=["ensembl_gene_id"])
    .copy()
)

print("BioMart gene info shape:", biomart_gene_info.shape)

BioMart gene info shape: (86369, 7)


In [49]:
fallback_rescued_positive = unmapped_positive_targets.merge(
    fallback_symbol_candidates,
    left_on="target_symbol",
    right_on="Gene Names (primary)",
    how="inner"
)

fallback_rescued_positive = fallback_rescued_positive.rename(columns={
    "Entry": "uniprot_accession",
    "Entry Name": "uniprot_entry_name",
    "Gene Names": "Gene Names",
    "Gene Names (primary)": "primary_gene_name",
    "Length": "Length"
})

# Gán ENSG từ Category A target_id
fallback_rescued_positive["ensembl_gene_id"] = fallback_rescued_positive["target_id"]

# Enrich gene-level coordinates/type từ BioMart
fallback_rescued_positive = fallback_rescued_positive.merge(
    biomart_gene_info,
    on="ensembl_gene_id",
    how="left"
)

# Các cột mapping-specific không có ở fallback ENST route
fallback_rescued_positive["n_unique_ENSG_per_uniprot"] = np.nan
fallback_rescued_positive["uniprot_to_ensg_status"] = "rescued_by_exact_primary_gene_symbol"
fallback_rescued_positive["mapping_route"] = "fallback_exact_primary_gene_symbol"

print("Fallback rescued shape:", fallback_rescued_positive.shape)
fallback_rescued_positive[
    [
        "target_id",
        "target_symbol",
        "uniprot_accession",
        "uniprot_entry_name",
        "Length",
        "ensembl_gene_id",
        "ensembl_gene_name"
    ]
]

Fallback rescued shape: (5, 24)


,target_id,target_symbol,uniprot_accession,uniprot_entry_name,Length,ensembl_gene_id,ensembl_gene_name
0,ENSG00000166436,TRIM66,O15016,TRI66_HUMAN,1351,ENSG00000166436,TRIM66
1,ENSG00000175164,ABO,P16442,BGAT_HUMAN,354,ENSG00000175164,ABO
2,ENSG00000084734,GCKR,Q14397,GCKR_HUMAN,625,ENSG00000084734,GCKR
3,ENSG00000174326,SLC16A11,Q8NCK7,MOT11_HUMAN,447,ENSG00000174326,SLC16A11
4,ENSG00000060749,QSER1,Q2KHR3,QSER1_HUMAN,1735,ENSG00000060749,QSER1


In [50]:
direct_mapped_positive["mapping_route"] = "primary_ENST_to_ENSG"

In [51]:
final_columns = [
    # Category A label fields
    "target_id",
    "target_symbol",
    "target_name",
    "association_score",
    "positive_tier",
    "label",
    "label_name",
    "label_source",
    "label_rule",
    "label_version",

    # UniProt protein fields
    "uniprot_accession",
    "uniprot_entry_name",
    "Protein names",
    "Gene Names",
    "primary_gene_name",
    "Length",
    "sequence",
    "sequence_length_fasta",

    # Ensembl/gene coordinate fields
    "ensembl_gene_id",
    "ensembl_gene_name",
    "gene_type",
    "chromosome_or_scaffold",
    "gene_start_bp",
    "gene_end_bp",
    "strand",

    # mapping QA
    "n_unique_ENSG_per_uniprot",
    "uniprot_to_ensg_status",
    "mapping_route"
]

# Đảm bảo fallback có đủ cột
for col in final_columns:
    if col not in fallback_rescued_positive.columns:
        fallback_rescued_positive[col] = np.nan

for col in final_columns:
    if col not in direct_mapped_positive.columns:
        direct_mapped_positive[col] = np.nan

positive_combined_before_representative = pd.concat(
    [
        direct_mapped_positive[final_columns],
        fallback_rescued_positive[final_columns]
    ],
    ignore_index=True
)

print("Combined before representative selection:", positive_combined_before_representative.shape)
print("Unique targets:", positive_combined_before_representative["target_id"].nunique())

Combined before representative selection: (911, 28)
Unique targets: 910


In [52]:
positive_combined_before_representative["representative_rank_within_target"] = (
    positive_combined_before_representative
    .sort_values(
        ["target_id", "sequence_length_fasta", "uniprot_accession"],
        ascending=[True, False, True]
    )
    .groupby("target_id")
    .cumcount() + 1
)

positive_protein_ready_v1 = positive_combined_before_representative[
    positive_combined_before_representative["representative_rank_within_target"] == 1
].copy()

positive_protein_ready_v1["representative_sequence_rule"] = np.where(
    positive_protein_ready_v1["target_id"].eq("ENSG00000021645"),
    "multiple_reviewed_uniprot_sequences_select_longest",
    "single_selected_sequence"
)

print("Final protein-ready shape:", positive_protein_ready_v1.shape)
print("Final unique targets:", positive_protein_ready_v1["target_id"].nunique())

Final protein-ready shape: (910, 30)
Final unique targets: 910


In [53]:
positive_protein_ready_v1[
    positive_protein_ready_v1["target_symbol"] == "NRXN3"
][
    [
        "target_id",
        "target_symbol",
        "uniprot_accession",
        "uniprot_entry_name",
        "sequence_length_fasta",
        "representative_sequence_rule"
    ]
]

,target_id,target_symbol,uniprot_accession,uniprot_entry_name,sequence_length_fasta,representative_sequence_rule
398,ENSG00000021645,NRXN3,Q9Y4C0,NRX3A_HUMAN,1643.0,multiple_reviewed_uniprot_sequences_select_lon...


In [54]:
positive_mapping_audit_v1 = positive_combined_before_representative.copy()

positive_mapping_audit_v1["selected_for_protein_ready_v1"] = (
    positive_mapping_audit_v1["representative_rank_within_target"] == 1
)

positive_mapping_audit_v1.to_csv(
    output_dir / "t2d_positive_protein_mapping_audit_v1.csv",
    index=False
)

print("Audit shape:", positive_mapping_audit_v1.shape)

Audit shape: (911, 30)


In [55]:
positive_targets_excluded_from_protein_ready_v1 = (
    unmapped_positive_targets[
        ~unmapped_positive_targets["target_symbol"].isin(
            fallback_rescued_positive["target_symbol"]
        )
    ]
    .copy()
)

positive_targets_excluded_from_protein_ready_v1["exclusion_reason"] = (
    "No reviewed UniProt canonical protein sequence found via "
    "primary ENST-to-ENSG mapping or exact primary gene-symbol rescue"
)

positive_targets_excluded_from_protein_ready_v1.to_csv(
    output_dir / "t2d_positive_targets_excluded_from_protein_ready_v1.csv",
    index=False
)

positive_targets_excluded_from_protein_ready_v1

,target_id,target_symbol,target_name,association_score,positive_tier,exclusion_reason
892,ENSG00000293584,ENSG00000293584,novel protein,0.304489,Tier_B_strong_positive_0.30_to_0.50,No reviewed UniProt canonical protein sequence...


In [56]:
positive_protein_ready_v1.to_csv(
    output_dir / "t2d_positive_protein_ready_v1.csv",
    index=False
)

print("Saved:", output_dir / "t2d_positive_protein_ready_v1.csv")

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_positive_protein_ready_v1.csv


In [57]:
def write_positive_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['target_id']}|"
                f"{row['target_symbol']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"score={row['association_score']:.6f}|"
                f"tier={row['positive_tier']}"
            )
            sequence = str(row["sequence"])

            f.write(header + "\n")

            # wrap FASTA sequence at 60 chars per line
            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")

fasta_output_path = output_dir / "t2d_positive_protein_ready_v1.fasta"

write_positive_fasta(
    positive_protein_ready_v1,
    fasta_output_path
)

print("Saved:", fasta_output_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_positive_protein_ready_v1.fasta


In [58]:
final_validation = {
    "n_final_rows": len(positive_protein_ready_v1),
    "n_unique_targets": positive_protein_ready_v1["target_id"].nunique(),
    "n_unique_uniprot_accessions": positive_protein_ready_v1["uniprot_accession"].nunique(),
    "n_missing_sequences": positive_protein_ready_v1["sequence"].isna().sum(),
    "n_missing_uniprot_accessions": positive_protein_ready_v1["uniprot_accession"].isna().sum(),
    "n_fallback_symbol_rescued": (
        positive_protein_ready_v1["mapping_route"]
        == "fallback_exact_primary_gene_symbol"
    ).sum(),
    "n_primary_ENST_to_ENSG": (
        positive_protein_ready_v1["mapping_route"]
        == "primary_ENST_to_ENSG"
    ).sum(),
}

final_validation

{'n_final_rows': 910,
 'n_unique_targets': 910,
 'n_unique_uniprot_accessions': 910,
 'n_missing_sequences': 0,
 'n_missing_uniprot_accessions': 0,
 'n_fallback_symbol_rescued': 5,
 'n_primary_ENST_to_ENSG': 905}

In [60]:
ot_all_file = Path(
    r"D:\khoane\MAHE\Project\dataset\opentargets\opentargets_t2d_output\opentargets_t2d_associated_targets_all.csv"
)

df_ot_all = pd.read_csv(ot_all_file)

print("Open Targets all shape:", df_ot_all.shape)
print(df_ot_all.columns.tolist())
df_ot_all.head()

Open Targets all shape: (9957, 35)
['rank_by_score', 'disease_id', 'evidence_mode', 'target_id', 'target_symbol', 'target_name', 'association_score', 'ds_clinical_precedence', 'ds_gwas_credible_sets', 'ds_eva', 'ds_genomics_england', 'ds_eva_somatic', 'ds_europepmc', 'prio_geneticConstraint', 'prio_hasHighQualityChemicalProbes', 'prio_hasLigand', 'prio_hasPocket', 'prio_hasSafetyEvent', 'prio_hasSmallMoleculeBinder', 'prio_isInMembrane', 'prio_isSecreted', 'prio_maxClinicalStage', 'prio_mouseKOScore', 'prio_mouseOrthologMaxIdentityPercentage', 'prio_paralogMaxIdentityPercentage', 'prio_tissueDistribution', 'prio_tissueSpecificity', 'prio_geneEssentiality', 'ds_impc', 'ds_gene_burden', 'ds_uniprot_variants', 'ds_uniprot_literature', 'prio_isCancerDriverGene', 'ds_expression_atlas', 'prio_hasTEP']


,rank_by_score,disease_id,evidence_mode,target_id,target_symbol,target_name,association_score,ds_clinical_precedence,ds_gwas_credible_sets,ds_eva,...,prio_tissueDistribution,prio_tissueSpecificity,prio_geneEssentiality,ds_impc,ds_gene_burden,ds_uniprot_variants,ds_uniprot_literature,prio_isCancerDriverGene,ds_expression_atlas,prio_hasTEP
0,1,MONDO_0005148,direct_only,ENSG00000187486,KCNJ11,potassium inwardly rectifying channel subfamil...,0.865142,0.998500,0.940931,0.914573,...,0.0,0.50,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,MONDO_0005148,direct_only,ENSG00000006071,ABCC8,ATP binding cassette subfamily C member 8,0.864850,0.998500,0.908190,0.940676,...,0.0,0.75,0.0,0.604792,NaN,NaN,NaN,NaN,NaN,NaN
2,3,MONDO_0005148,direct_only,ENSG00000106633,GCK,glucokinase,0.861234,0.609747,0.736958,0.903364,...,0.0,0.50,0.0,0.740716,0.990233,0.607931,0.607931,NaN,NaN,NaN
3,4,MONDO_0005148,direct_only,ENSG00000132170,PPARG,peroxisome proliferator activated receptor gamma,0.848609,0.997083,0.884298,0.847504,...,0.0,0.50,0.0,0.573665,0.261788,NaN,NaN,255.0,NaN,NaN
4,5,MONDO_0005148,direct_only,ENSG00000171105,INSR,insulin receptor,0.788737,0.997991,0.736325,0.616037,...,-1.0,-1.00,0.0,0.551506,NaN,0.413731,0.303965,NaN,NaN,NaN


In [61]:
df_ot_unique = (
    df_ot_all
    .sort_values("association_score", ascending=False)
    .drop_duplicates(subset=["target_id"], keep="first")
    .copy()
)

print("Unique Open Targets targets:", df_ot_unique.shape)
print("Unique target IDs:", df_ot_unique["target_id"].nunique())

Unique Open Targets targets: (9687, 35)
Unique target IDs: 9687


In [62]:
positive_target_ids = set(
    df_ot_unique.loc[
        df_ot_unique["association_score"] >= 0.30,
        "target_id"
    ]
)

grey_target_ids = set(
    df_ot_unique.loc[
        (df_ot_unique["association_score"] > 0) &
        (df_ot_unique["association_score"] < 0.30),
        "target_id"
    ]
)

ot_zero_score_target_ids = set(
    df_ot_unique.loc[
        df_ot_unique["association_score"] == 0,
        "target_id"
    ]
)

print("Positive OT target IDs:", len(positive_target_ids))
print("Grey OT target IDs:", len(grey_target_ids))
print("Zero-score OT target IDs:", len(ot_zero_score_target_ids))

Positive OT target IDs: 911
Grey OT target IDs: 8775
Zero-score OT target IDs: 1


In [63]:
negative_pool_base = protein_ensg_map_raw.copy()

print("Negative base raw shape:", negative_pool_base.shape)
print("Unique ENSG:", negative_pool_base["ensembl_gene_id"].nunique())
print("Unique UniProt:", negative_pool_base["uniprot_accession"].nunique())

Negative base raw shape: (21811, 17)
Unique ENSG: 21784
Unique UniProt: 19351


In [64]:
negative_after_remove_positive = negative_pool_base[
    ~negative_pool_base["ensembl_gene_id"].isin(positive_target_ids)
].copy()

print("After removing positive ENSG:", negative_after_remove_positive.shape)
print("Unique ENSG:", negative_after_remove_positive["ensembl_gene_id"].nunique())

After removing positive ENSG: (20905, 17)
Unique ENSG: 20879


In [65]:
negative_after_remove_grey = negative_after_remove_positive[
    ~negative_after_remove_positive["ensembl_gene_id"].isin(grey_target_ids)
].copy()

print("After removing positive + grey ENSG:", negative_after_remove_grey.shape)
print("Unique ENSG:", negative_after_remove_grey["ensembl_gene_id"].nunique())
print("Unique UniProt:", negative_after_remove_grey["uniprot_accession"].nunique())

After removing positive + grey ENSG: (12500, 17)
Unique ENSG: 12488
Unique UniProt: 10584


In [66]:
positive_uniprot_accessions = set(
    positive_protein_ready_v1["uniprot_accession"]
)

print("Positive UniProt accessions:", len(positive_uniprot_accessions))

Positive UniProt accessions: 910


In [67]:
negative_after_remove_positive_accessions = negative_after_remove_grey[
    ~negative_after_remove_grey["uniprot_accession"].isin(positive_uniprot_accessions)
].copy()

print("After removing positive UniProt accessions:", negative_after_remove_positive_accessions.shape)
print("Unique ENSG:", negative_after_remove_positive_accessions["ensembl_gene_id"].nunique())
print("Unique UniProt:", negative_after_remove_positive_accessions["uniprot_accession"].nunique())

After removing positive UniProt accessions: (12461, 17)
Unique ENSG: 12449
Unique UniProt: 10556


In [68]:
ambiguous_positive_uniprot_accessions = set(
    positive_using_ambiguous_uniprot["uniprot_accession"]
)

print("Ambiguous positive UniProt accessions:", len(ambiguous_positive_uniprot_accessions))

Ambiguous positive UniProt accessions: 28


In [69]:
negative_strict_candidate_raw = negative_after_remove_positive_accessions[
    ~negative_after_remove_positive_accessions["uniprot_accession"].isin(
        ambiguous_positive_uniprot_accessions
    )
].copy()

print("Strict negative raw shape:", negative_strict_candidate_raw.shape)
print("Unique ENSG:", negative_strict_candidate_raw["ensembl_gene_id"].nunique())
print("Unique UniProt:", negative_strict_candidate_raw["uniprot_accession"].nunique())

Strict negative raw shape: (12461, 17)
Unique ENSG: 12449
Unique UniProt: 10556


In [70]:
negative_strict_candidate_raw["gene_type"].value_counts(dropna=False).head(30)

gene_type
protein_coding    12112
IG_V_gene           186
TR_V_gene           125
TR_J_gene            21
IG_C_gene             8
IG_J_gene             4
TR_D_gene             3
IG_D_gene             2
Name: count, dtype: int64

In [73]:
negative_uniprot_to_ensg_counts = (
    negative_strict_candidate_raw
    .groupby("uniprot_accession")["ensembl_gene_id"]
    .nunique()
    .reset_index(name="n_unique_ENSG_per_uniprot_in_negative")
)

negative_uniprot_to_ensg_counts[
    "n_unique_ENSG_per_uniprot_in_negative"
].value_counts().sort_index()

n_unique_ENSG_per_uniprot_in_negative
1     9616
2      669
3       70
4       47
5       35
6       36
7       42
8       19
9        5
10       8
12       2
13       2
14       1
17       1
21       1
22       1
23       1
Name: count, dtype: int64

In [74]:
negative_ensg_to_uniprot_counts = (
    negative_strict_candidate_raw
    .groupby("ensembl_gene_id")["uniprot_accession"]
    .nunique()
    .reset_index(name="n_unique_uniprot_per_ENSG_in_negative")
)

negative_ensg_to_uniprot_counts[
    "n_unique_uniprot_per_ENSG_in_negative"
].value_counts().sort_index()

n_unique_uniprot_per_ENSG_in_negative
1    12437
2       12
Name: count, dtype: int64

In [75]:
negative_step1_protein_coding = negative_strict_candidate_raw[
    negative_strict_candidate_raw["gene_type"] == "protein_coding"
].copy()

print("After keeping protein_coding only:", negative_step1_protein_coding.shape)
print("Unique ENSG:", negative_step1_protein_coding["ensembl_gene_id"].nunique())
print("Unique UniProt:", negative_step1_protein_coding["uniprot_accession"].nunique())

After keeping protein_coding only: (12112, 17)
Unique ENSG: 12100
Unique UniProt: 10308


In [76]:
neg_uniprot_to_ensg_after_pc = (
    negative_step1_protein_coding
    .groupby("uniprot_accession")["ensembl_gene_id"]
    .nunique()
    .reset_index(name="n_unique_ENSG_per_uniprot")
)

neg_uniprot_to_ensg_after_pc[
    "n_unique_ENSG_per_uniprot"
].value_counts().sort_index()

n_unique_ENSG_per_uniprot
1     9469
2      568
3       70
4       47
5       35
6       36
7       42
8       19
9        5
10       8
12       2
13       2
14       1
17       1
21       1
22       1
23       1
Name: count, dtype: int64

In [77]:
clean_negative_uniprot_accessions = set(
    neg_uniprot_to_ensg_after_pc.loc[
        neg_uniprot_to_ensg_after_pc["n_unique_ENSG_per_uniprot"] == 1,
        "uniprot_accession"
    ]
)

negative_step2_one_uniprot_one_ensg = negative_step1_protein_coding[
    negative_step1_protein_coding["uniprot_accession"].isin(
        clean_negative_uniprot_accessions
    )
].copy()

print("After keeping UniProt accession mapping to exactly 1 ENSG:",
      negative_step2_one_uniprot_one_ensg.shape)
print("Unique ENSG:",
      negative_step2_one_uniprot_one_ensg["ensembl_gene_id"].nunique())
print("Unique UniProt:",
      negative_step2_one_uniprot_one_ensg["uniprot_accession"].nunique())

After keeping UniProt accession mapping to exactly 1 ENSG: (9469, 17)
Unique ENSG: 9459
Unique UniProt: 9469


In [78]:
neg_ensg_to_uniprot_after_clean = (
    negative_step2_one_uniprot_one_ensg
    .groupby("ensembl_gene_id")["uniprot_accession"]
    .nunique()
    .reset_index(name="n_unique_uniprot_per_ENSG")
)

neg_ensg_to_uniprot_after_clean[
    "n_unique_uniprot_per_ENSG"
].value_counts().sort_index()

n_unique_uniprot_per_ENSG
1    9449
2      10
Name: count, dtype: int64

In [79]:
negative_multi_uniprot_per_ensg = neg_ensg_to_uniprot_after_clean[
    neg_ensg_to_uniprot_after_clean["n_unique_uniprot_per_ENSG"] > 1
]

print("ENSG with >1 UniProt after cleaning:",
      len(negative_multi_uniprot_per_ensg))

negative_multi_uniprot_per_ensg.head(30)

ENSG with >1 UniProt after cleaning: 10


,ensembl_gene_id,n_unique_uniprot_per_ENSG
1494,ENSG00000109113,2
1952,ENSG00000118507,2
4442,ENSG00000160404,2
4800,ENSG00000164172,2
5474,ENSG00000169905,2
6353,ENSG00000179915,2
6960,ENSG00000186184,2
8186,ENSG00000221887,2
8244,ENSG00000225830,2
9123,ENSG00000282841,2


In [80]:
negative_step2_one_uniprot_one_ensg = negative_step2_one_uniprot_one_ensg.sort_values(
    by=["ensembl_gene_id", "sequence_length_fasta", "uniprot_accession"],
    ascending=[True, False, True]
).copy()

negative_step2_one_uniprot_one_ensg["representative_rank_within_ENSG"] = (
    negative_step2_one_uniprot_one_ensg
    .groupby("ensembl_gene_id")
    .cumcount() + 1
)

negative_candidate_pool_clean_v1 = negative_step2_one_uniprot_one_ensg[
    negative_step2_one_uniprot_one_ensg["representative_rank_within_ENSG"] == 1
].copy()

negative_candidate_pool_clean_v1["label"] = 0
negative_candidate_pool_clean_v1["label_name"] = "T2D_negative_candidate"
negative_candidate_pool_clean_v1["negative_pool_version"] = "Negative_Candidate_Pool_Clean_v1"
negative_candidate_pool_clean_v1["negative_rule"] = (
    "Reviewed human UniProt protein; protein_coding; not Category A positive; "
    "not Open Targets grey-zone 0<score<0.30; not positive UniProt accession; "
    "UniProt maps exactly one ENSG; one representative sequence per ENSG"
)

print("Negative Candidate Pool Clean v1 shape:",
      negative_candidate_pool_clean_v1.shape)
print("Unique ENSG:",
      negative_candidate_pool_clean_v1["ensembl_gene_id"].nunique())
print("Unique UniProt:",
      negative_candidate_pool_clean_v1["uniprot_accession"].nunique())

Negative Candidate Pool Clean v1 shape: (9459, 22)
Unique ENSG: 9459
Unique UniProt: 9459


In [81]:
negative_clean_validation = {
    "n_final_negative_pool_rows": len(negative_candidate_pool_clean_v1),
    "n_unique_negative_ENSG": negative_candidate_pool_clean_v1["ensembl_gene_id"].nunique(),
    "n_unique_negative_uniprot": negative_candidate_pool_clean_v1["uniprot_accession"].nunique(),
    "n_missing_sequences": int(negative_candidate_pool_clean_v1["sequence"].isna().sum()),
    "n_missing_uniprot_accessions": int(
        negative_candidate_pool_clean_v1["uniprot_accession"].isna().sum()
    ),
    "n_remaining_positive_ENSG_overlap": int(
        negative_candidate_pool_clean_v1["ensembl_gene_id"].isin(
            positive_target_ids
        ).sum()
    ),
    "n_remaining_grey_ENSG_overlap": int(
        negative_candidate_pool_clean_v1["ensembl_gene_id"].isin(
            grey_target_ids
        ).sum()
    ),
    "n_remaining_positive_uniprot_overlap": int(
        negative_candidate_pool_clean_v1["uniprot_accession"].isin(
            positive_uniprot_accessions
        ).sum()
    )
}

negative_clean_validation

{'n_final_negative_pool_rows': 9459,
 'n_unique_negative_ENSG': 9459,
 'n_unique_negative_uniprot': 9459,
 'n_missing_sequences': 0,
 'n_missing_uniprot_accessions': 0,
 'n_remaining_positive_ENSG_overlap': 0,
 'n_remaining_grey_ENSG_overlap': 0,
 'n_remaining_positive_uniprot_overlap': 0}

In [83]:
negative_candidate_pool_clean_v1.to_csv(
    output_dir / "t2d_negative_candidate_pool_clean_v1.csv",
    index=False
)

print("Saved:",
      output_dir / "t2d_negative_candidate_pool_clean_v1.csv")

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_candidate_pool_clean_v1.csv


In [84]:
def write_negative_pool_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['ensembl_gene_id']}|"
                f"{row['ensembl_gene_name']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"label=0"
            )

            sequence = str(row["sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")

negative_fasta_path = output_dir / "t2d_negative_candidate_pool_clean_v1.fasta"

write_negative_pool_fasta(
    negative_candidate_pool_clean_v1,
    negative_fasta_path
)

print("Saved:", negative_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_candidate_pool_clean_v1.fasta


In [82]:
positive_length_summary = positive_protein_ready_v1["sequence_length_fasta"].describe()
positive_length_summary

count      910.000000
mean       793.135165
std       1300.610420
min         41.000000
25%        354.250000
50%        561.000000
75%        948.500000
max      34350.000000
Name: sequence_length_fasta, dtype: float64

In [85]:
negative_length_summary = negative_candidate_pool_clean_v1["sequence_length_fasta"].describe()
negative_length_summary

count    9459.000000
mean      543.400254
std       512.781117
min        25.000000
25%       256.000000
50%       411.000000
75%       658.500000
max      7968.000000
Name: sequence_length_fasta, dtype: float64

In [86]:
positive_for_sampling = positive_protein_ready_v1.copy()
negative_for_sampling = negative_candidate_pool_clean_v1.copy()

# Tạo bin theo quantile của positive protein lengths
positive_for_sampling["length_bin"], bin_edges = pd.qcut(
    positive_for_sampling["sequence_length_fasta"],
    q=10,
    retbins=True,
    duplicates="drop"
)

print("Bin edges:")
print(bin_edges)

positive_for_sampling["length_bin"].value_counts().sort_index()

Bin edges:
[   41.    210.    315.    398.    469.    561.    683.2   852.6  1078.
  1447.3 34350. ]


length_bin
(40.999, 210.0]      92
(210.0, 315.0]       91
(315.0, 398.0]       92
(398.0, 469.0]       91
(469.0, 561.0]       90
(561.0, 683.2]       90
(683.2, 852.6]       91
(852.6, 1078.0]      92
(1078.0, 1447.3]     90
(1447.3, 34350.0]    91
Name: count, dtype: int64

In [87]:
negative_for_sampling["length_bin"] = pd.cut(
    negative_for_sampling["sequence_length_fasta"],
    bins=bin_edges,
    include_lowest=True
)

# Xem negative pool có đủ candidate trong từng bin không
negative_bin_counts = (
    negative_for_sampling["length_bin"]
    .value_counts()
    .sort_index()
)

positive_bin_counts = (
    positive_for_sampling["length_bin"]
    .value_counts()
    .sort_index()
)

sampling_bin_check = pd.DataFrame({
    "positive_count_needed": positive_bin_counts,
    "negative_candidates_available": negative_bin_counts
})

sampling_bin_check

,positive_count_needed,negative_candidates_available
length_bin,,
"(40.999, 210.0]",92,1689
"(210.0, 315.0]",91,1661
"(315.0, 398.0]",92,1231
"(398.0, 469.0]",91,825
"(469.0, 561.0]",90,964
"(561.0, 683.2]",90,872
"(683.2, 852.6]",91,789
"(852.6, 1078.0]",92,578
"(1078.0, 1447.3]",90,417


In [88]:
sampled_negative_parts = []

random_seed = 42

for bin_label, n_needed in positive_bin_counts.items():
    bin_candidates = negative_for_sampling[
        negative_for_sampling["length_bin"] == bin_label
    ]

    sampled_bin = bin_candidates.sample(
        n=n_needed,
        random_state=random_seed
    )

    sampled_negative_parts.append(sampled_bin)

negative_balanced_matched_v1 = pd.concat(
    sampled_negative_parts,
    ignore_index=True
)

print("Balanced matched negative shape:", negative_balanced_matched_v1.shape)
print("Unique ENSG:", negative_balanced_matched_v1["ensembl_gene_id"].nunique())
print("Unique UniProt:", negative_balanced_matched_v1["uniprot_accession"].nunique())

Balanced matched negative shape: (910, 23)
Unique ENSG: 910
Unique UniProt: 910


In [89]:
negative_balanced_matched_v1.to_csv(
    output_dir / "t2d_negative_balanced_lengthmatched_v1.csv",
    index=False
)

print("Saved final negative set.")

Saved final negative set.


In [91]:
def write_negative_final_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['ensembl_gene_id']}|"
                f"{row['ensembl_gene_name']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"label=0"
            )

            sequence = str(row["sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")

negative_final_fasta_path = (
    output_dir / "t2d_negative_balanced_lengthmatched_v1.fasta"
)

write_negative_final_fasta(
    negative_balanced_matched_v1,
    negative_final_fasta_path
)

print("Saved FASTA:", negative_final_fasta_path)

Saved FASTA: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_balanced_lengthmatched_v1.fasta


In [92]:
negative_balanced_matched_v1

,uniprot_accession,uniprot_entry_name,Protein names,Gene Names,primary_gene_name,Length,sequence,sequence_length_fasta,ensembl_gene_id,ensembl_gene_name,...,gene_end_bp,strand,n_unique_ENSG_per_uniprot,uniprot_to_ensg_status,representative_rank_within_ENSG,label,label_name,negative_pool_version,negative_rule,length_bin
0,Q5T5B0,LCE3E_HUMAN,Late cornified envelope protein 3E (Late envel...,LCE3E LEP17,LCE3E,92,MSCQQNQKQCQPPPKCPSPKCPPKNPVQCLPPASSGCAPSSGGCGP...,92,ENSG00000185966,LCE3E,...,152566780,-1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]"
1,A0A286YF46,SCGR5_HUMAN,Small cysteine and glycine repeat-containing p...,SCYGR5 KRTAP28-5,SCYGR5,85,MGCCGCGGCGGCGGGCGGGCGSCTTCRCYRVGCCSSCCPCCRGCCG...,85,ENSG00000284667,SCYGR5,...,227667061,1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]"
2,Q8TDE3,RNAS8_HUMAN,Ribonuclease 8 (RNase 8) (EC 3.1.27.-),RNASE8,RNASE8,154,MAPARAGCCPLLLLLLGLWVAEVLVRAKPKDMTSSQWFKTQHVQPS...,154,ENSG00000173431,RNASE8,...,21058455,1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]"
3,Q3LI63,KR201_HUMAN,Keratin-associated protein 20-1,KRTAP20-1 KAP20.1,KRTAP20-1,56,MIYYSNYYGGYGYGGLGCGYGCGYRGYGCGYGGYGGYGNGYYCPSC...,56,ENSG00000244624,KRTAP20-1,...,30616699,1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]"
4,P41567,EIF1_HUMAN,Eukaryotic translation initiation factor 1 (eI...,EIF1 SUI1,EIF1,113,MSAIQNLHSFDPFADASKGDDLLPAGTEDYIHIRIQQRNGRKTLTT...,113,ENSG00000173812,EIF1,...,41692668,1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
905,Q9BZ29,DOCK9_HUMAN,Dedicator of cytokinesis protein 9 (Cdc42 guan...,DOCK9 KIAA1058 ZIZ1,DOCK9,2069,MSQPPLLPASAETRKFTRALSKPGTAAELRQSVSEVVRGSVLLAKP...,2069,ENSG00000088387,DOCK9,...,99086625,-1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(1447.3, 34350.0]"
906,Q9UHC1,MLH3_HUMAN,DNA mismatch repair protein Mlh3 (MutL protein...,MLH3,MLH3,1453,MIKCLSVEVQAKLRSGLAISSLGQCVEELALNSIDAEAKCVAVRVN...,1453,ENSG00000119684,MLH3,...,75051532,-1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(1447.3, 34350.0]"
907,Q96JM2,ZN462_HUMAN,Zinc finger protein 462 (Zinc finger PBX1-inte...,ZNF462 KIAA1803,ZNF462,2506,MEVLQCDGCDFRAPSYEDLKAHIQDVHTAFLQPTDVAEDNVNELRC...,2506,ENSG00000148143,ZNF462,...,107013634,1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(1447.3, 34350.0]"
908,Q8IZY2,ABCA7_HUMAN,Phospholipid-transporting ATPase ABCA7 (EC 7.6...,ABCA7,ABCA7,2146,MAFWTQLMLLLWKNFMYRRRQPVQLLVELLWPLFLFFILVAVRHSH...,2146,ENSG00000064687,ABCA7,...,1065572,1,1,one_uniprot_to_one_ensg,1,0,T2D_negative_candidate,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(1447.3, 34350.0]"


In [93]:
negative_balanced_matched_v1["label"] = 0
negative_balanced_matched_v1["label_name"] = "T2D_negative"
negative_balanced_matched_v1["negative_sample_version"] = (
    "Negative_Balanced_LengthMatched_v1"
)
negative_balanced_matched_v1["negative_sampling_rule"] = (
    "Sampled from Negative Candidate Pool Clean v1 using 1:1 "
    "positive-negative balance with sequence-length-stratified sampling "
    "based on positive-set decile bins; random_state=42"
)

negative_balanced_matched_v1.head()

,uniprot_accession,uniprot_entry_name,Protein names,Gene Names,primary_gene_name,Length,sequence,sequence_length_fasta,ensembl_gene_id,ensembl_gene_name,...,n_unique_ENSG_per_uniprot,uniprot_to_ensg_status,representative_rank_within_ENSG,label,label_name,negative_pool_version,negative_rule,length_bin,negative_sample_version,negative_sampling_rule
0,Q5T5B0,LCE3E_HUMAN,Late cornified envelope protein 3E (Late envel...,LCE3E LEP17,LCE3E,92,MSCQQNQKQCQPPPKCPSPKCPPKNPVQCLPPASSGCAPSSGGCGP...,92,ENSG00000185966,LCE3E,...,1,one_uniprot_to_one_ensg,1,0,T2D_negative,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]",Negative_Balanced_LengthMatched_v1,Sampled from Negative Candidate Pool Clean v1 ...
1,A0A286YF46,SCGR5_HUMAN,Small cysteine and glycine repeat-containing p...,SCYGR5 KRTAP28-5,SCYGR5,85,MGCCGCGGCGGCGGGCGGGCGSCTTCRCYRVGCCSSCCPCCRGCCG...,85,ENSG00000284667,SCYGR5,...,1,one_uniprot_to_one_ensg,1,0,T2D_negative,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]",Negative_Balanced_LengthMatched_v1,Sampled from Negative Candidate Pool Clean v1 ...
2,Q8TDE3,RNAS8_HUMAN,Ribonuclease 8 (RNase 8) (EC 3.1.27.-),RNASE8,RNASE8,154,MAPARAGCCPLLLLLLGLWVAEVLVRAKPKDMTSSQWFKTQHVQPS...,154,ENSG00000173431,RNASE8,...,1,one_uniprot_to_one_ensg,1,0,T2D_negative,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]",Negative_Balanced_LengthMatched_v1,Sampled from Negative Candidate Pool Clean v1 ...
3,Q3LI63,KR201_HUMAN,Keratin-associated protein 20-1,KRTAP20-1 KAP20.1,KRTAP20-1,56,MIYYSNYYGGYGYGGLGCGYGCGYRGYGCGYGGYGGYGNGYYCPSC...,56,ENSG00000244624,KRTAP20-1,...,1,one_uniprot_to_one_ensg,1,0,T2D_negative,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]",Negative_Balanced_LengthMatched_v1,Sampled from Negative Candidate Pool Clean v1 ...
4,P41567,EIF1_HUMAN,Eukaryotic translation initiation factor 1 (eI...,EIF1 SUI1,EIF1,113,MSAIQNLHSFDPFADASKGDDLLPAGTEDYIHIRIQQRNGRKTLTT...,113,ENSG00000173812,EIF1,...,1,one_uniprot_to_one_ensg,1,0,T2D_negative,Negative_Candidate_Pool_Clean_v1,Reviewed human UniProt protein; protein_coding...,"(40.999, 210.0]",Negative_Balanced_LengthMatched_v1,Sampled from Negative Candidate Pool Clean v1 ...


In [94]:
length_distribution_check = pd.DataFrame({
    "positive_count": (
        positive_for_sampling["length_bin"]
        .value_counts()
        .sort_index()
    ),
    "sampled_negative_count": (
        negative_balanced_matched_v1["length_bin"]
        .value_counts()
        .sort_index()
    )
})

length_distribution_check

,positive_count,sampled_negative_count
length_bin,,
"(40.999, 210.0]",92,92
"(210.0, 315.0]",91,91
"(315.0, 398.0]",92,92
"(398.0, 469.0]",91,91
"(469.0, 561.0]",90,90
"(561.0, 683.2]",90,90
"(683.2, 852.6]",91,91
"(852.6, 1078.0]",92,92
"(1078.0, 1447.3]",90,90


In [95]:
pd.DataFrame({
    "positive_length": positive_protein_ready_v1["sequence_length_fasta"].describe(),
    "sampled_negative_length": negative_balanced_matched_v1["sequence_length_fasta"].describe()
})

,positive_length,sampled_negative_length
count,910.000000,910.000000
mean,793.135165,733.528571
std,1300.610420,571.342804
min,41.000000,56.000000
25%,354.250000,351.250000
50%,561.000000,560.500000
75%,948.500000,932.000000
max,34350.000000,4516.000000


In [96]:
negative_balanced_validation = {
    "n_negative_rows": len(negative_balanced_matched_v1),
    "n_unique_negative_ENSG": negative_balanced_matched_v1["ensembl_gene_id"].nunique(),
    "n_unique_negative_uniprot": negative_balanced_matched_v1["uniprot_accession"].nunique(),
    "n_missing_sequences": int(
        negative_balanced_matched_v1["sequence"].isna().sum()
    ),
    "n_overlap_positive_ENSG": int(
        negative_balanced_matched_v1["ensembl_gene_id"].isin(
            positive_protein_ready_v1["target_id"]
        ).sum()
    ),
    "n_overlap_positive_uniprot": int(
        negative_balanced_matched_v1["uniprot_accession"].isin(
            positive_protein_ready_v1["uniprot_accession"]
        ).sum()
    ),
    "label_0_count": int(
        (negative_balanced_matched_v1["label"] == 0).sum()
    )
}

negative_balanced_validation

{'n_negative_rows': 910,
 'n_unique_negative_ENSG': 910,
 'n_unique_negative_uniprot': 910,
 'n_missing_sequences': 0,
 'n_overlap_positive_ENSG': 0,
 'n_overlap_positive_uniprot': 0,
 'label_0_count': 910}

In [97]:
negative_balanced_matched_v1.to_csv(
    output_dir / "t2d_negative_balanced_lengthmatched_v1.csv",
    index=False
)

print("Saved:", output_dir / "t2d_negative_balanced_lengthmatched_v1.csv")

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_balanced_lengthmatched_v1.csv


In [98]:
def write_negative_final_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['ensembl_gene_id']}|"
                f"{row['ensembl_gene_name']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"label=0"
            )

            sequence = str(row["sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")

negative_final_fasta_path = (
    output_dir / "t2d_negative_balanced_lengthmatched_v1.fasta"
)

write_negative_final_fasta(
    negative_balanced_matched_v1,
    negative_final_fasta_path
)

print("Saved:", negative_final_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_balanced_lengthmatched_v1.fasta


In [99]:
positive_model_v1 = positive_protein_ready_v1.copy()

positive_model_v1["gene_id"] = positive_model_v1["target_id"]
positive_model_v1["gene_symbol"] = positive_model_v1["target_symbol"]
positive_model_v1["gene_name"] = positive_model_v1["target_name"]

positive_model_v1["class_source"] = "Category_A_T2D_positive_label"
positive_model_v1["dataset_role"] = "positive"
positive_model_v1["classification_dataset_version"] = (
    "T2D_Protein_Classification_Balanced_v1"
)

print("Positive model shape:", positive_model_v1.shape)
positive_model_v1[
    [
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "sequence_length_fasta",
        "label"
    ]
].head()

Positive model shape: (910, 36)


,gene_id,gene_symbol,uniprot_accession,sequence_length_fasta,label
0,ENSG00000187486,KCNJ11,Q14654,390.0,1.0
1,ENSG00000006071,ABCC8,Q09428,1581.0,1.0
2,ENSG00000106633,GCK,P35557,465.0,1.0
3,ENSG00000132170,PPARG,P37231,505.0,1.0
4,ENSG00000171105,INSR,P06213,1382.0,1.0


In [100]:
negative_model_v1 = negative_balanced_matched_v1.copy()

negative_model_v1["gene_id"] = negative_model_v1["ensembl_gene_id"]
negative_model_v1["gene_symbol"] = negative_model_v1["ensembl_gene_name"]
negative_model_v1["gene_name"] = negative_model_v1["ensembl_gene_name"]

negative_model_v1["class_source"] = (
    "Category_B_negative_clean_pool_lengthmatched_sampling"
)
negative_model_v1["dataset_role"] = "negative"
negative_model_v1["classification_dataset_version"] = (
    "T2D_Protein_Classification_Balanced_v1"
)

print("Negative model shape:", negative_model_v1.shape)
negative_model_v1[
    [
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "sequence_length_fasta",
        "label"
    ]
].head()

Negative model shape: (910, 31)


,gene_id,gene_symbol,uniprot_accession,sequence_length_fasta,label
0,ENSG00000185966,LCE3E,Q5T5B0,92,0
1,ENSG00000284667,SCYGR5,A0A286YF46,85,0
2,ENSG00000173431,RNASE8,Q8TDE3,154,0
3,ENSG00000244624,KRTAP20-1,Q3LI63,56,0
4,ENSG00000173812,EIF1,P41567,113,0


In [101]:
model_final_columns = [
    # Core identity
    "gene_id",
    "gene_symbol",
    "gene_name",
    "uniprot_accession",
    "uniprot_entry_name",

    # Protein input
    "sequence",
    "sequence_length_fasta",

    # Label
    "label",
    "label_name",
    "dataset_role",
    "class_source",
    "classification_dataset_version",

    # Positive-only evidence fields
    "association_score",
    "positive_tier",
    "label_source",
    "label_rule",
    "label_version",

    # Negative-only fields
    "negative_sample_version",
    "negative_sampling_rule",

    # Sequence/mapping audit
    "mapping_route",
    "uniprot_to_ensg_status",
    "representative_sequence_rule",
    "length_bin",

    # Genomic / mapping metadata
    "ensembl_gene_id",
    "ensembl_gene_name",
    "gene_type",
    "chromosome_or_scaffold",
    "gene_start_bp",
    "gene_end_bp",
    "strand"
]

In [102]:
for col in model_final_columns:
    if col not in positive_model_v1.columns:
        positive_model_v1[col] = np.nan

for col in model_final_columns:
    if col not in negative_model_v1.columns:
        negative_model_v1[col] = np.nan

In [103]:
t2d_protein_classification_dataset_balanced_v1 = pd.concat(
    [
        positive_model_v1[model_final_columns],
        negative_model_v1[model_final_columns]
    ],
    ignore_index=True
)

print(
    "Final classification dataset shape:",
    t2d_protein_classification_dataset_balanced_v1.shape
)

t2d_protein_classification_dataset_balanced_v1["label"].value_counts()

Final classification dataset shape: (1820, 30)


C:\Users\khoah\AppData\Local\Temp\ipykernel_21600\943918974.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  t2d_protein_classification_dataset_balanced_v1 = pd.concat(


label
0.0    910
1.0    905
Name: count, dtype: int64

In [104]:
t2d_protein_classification_dataset_balanced_v1 = (
    t2d_protein_classification_dataset_balanced_v1
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

t2d_protein_classification_dataset_balanced_v1.head()

,gene_id,gene_symbol,gene_name,uniprot_accession,uniprot_entry_name,sequence,sequence_length_fasta,label,label_name,dataset_role,...,uniprot_to_ensg_status,representative_sequence_rule,length_bin,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,ENSG00000084092,NOA1,NOA1,Q8NC60,NOA1_HUMAN,MLPARLPFRLLSLFLRGSAPTAARHGLREPLLERRCAAASSFQHSS...,698.0,0.0,T2D_negative,negative,...,one_uniprot_to_one_ensg,NaN,"(683.2, 852.6]",ENSG00000084092,NOA1,protein_coding,4,56962876.0,56977623.0,-1.0
1,ENSG00000185716,MOSMO,MOSMO,Q8NHV5,MOSMO_HUMAN,MDKLTIISGCLFLAADIFAIASIANPDWINTGESAGALTVGLVRQC...,167.0,0.0,T2D_negative,negative,...,one_uniprot_to_one_ensg,NaN,"(40.999, 210.0]",ENSG00000185716,MOSMO,protein_coding,16,22007638.0,22087534.0,1.0
2,ENSG00000106113,CRHR2,corticotropin releasing hormone receptor 2,Q13324,CRFR2_HUMAN,MDAALLHSLLEANCSLALAEELLLDGWGPPLDPEGPYSYCNTTLDQ...,411.0,1.0,T2D_positive,positive,...,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000106113,CRHR2,protein_coding,7,30651444.0,30700129.0,-1.0
3,ENSG00000165392,WRN,WRN RecQ like helicase,Q14191,WRN_HUMAN,MSEKKLETTAQQRKCPEWMNVQNKRCAVEERKACVRKSVFEDDLPF...,1432.0,1.0,T2D_positive,positive,...,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000165392,WRN,protein_coding,8,31033779.0,31176138.0,1.0
4,ENSG00000131196,NFATC1,nuclear factor of activated T cells 1,O95644,NFAC1_HUMAN,MPSTSFPVPSKFPLGPAAAVFGRGETLGPAPRAGGTMKSAEEEHYG...,943.0,1.0,T2D_positive,positive,...,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000131196,NFATC1,protein_coding,18,79395856.0,79529325.0,1.0


In [105]:
final_classification_validation = {
    "n_total_rows": len(t2d_protein_classification_dataset_balanced_v1),
    "n_positive_rows": int(
        (t2d_protein_classification_dataset_balanced_v1["label"] == 1).sum()
    ),
    "n_negative_rows": int(
        (t2d_protein_classification_dataset_balanced_v1["label"] == 0).sum()
    ),
    "n_unique_gene_ids": (
        t2d_protein_classification_dataset_balanced_v1["gene_id"].nunique()
    ),
    "n_unique_uniprot_accessions": (
        t2d_protein_classification_dataset_balanced_v1[
            "uniprot_accession"
        ].nunique()
    ),
    "n_missing_sequences": int(
        t2d_protein_classification_dataset_balanced_v1["sequence"]
        .isna()
        .sum()
    ),
    "n_duplicate_gene_ids": int(
        t2d_protein_classification_dataset_balanced_v1["gene_id"]
        .duplicated()
        .sum()
    ),
    "n_duplicate_uniprot_accessions": int(
        t2d_protein_classification_dataset_balanced_v1[
            "uniprot_accession"
        ]
        .duplicated()
        .sum()
    ),
}

final_classification_validation

{'n_total_rows': 1820,
 'n_positive_rows': 905,
 'n_negative_rows': 910,
 'n_unique_gene_ids': 1820,
 'n_unique_uniprot_accessions': 1820,
 'n_missing_sequences': 0,
 'n_duplicate_gene_ids': 0,
 'n_duplicate_uniprot_accessions': 0}

In [106]:
dataset_v1_summary = pd.DataFrame([
    {
        "dataset": "Positive Protein-Ready v1",
        "n_rows": len(positive_protein_ready_v1),
        "label": 1,
        "description": "T2D-positive reviewed human UniProt protein sequences"
    },
    {
        "dataset": "Negative Candidate Pool Clean v1",
        "n_rows": len(negative_candidate_pool_clean_v1),
        "label": "candidate_pool",
        "description": "Strict negative/background candidate pool before balanced sampling"
    },
    {
        "dataset": "Negative Balanced Length-Matched v1",
        "n_rows": len(negative_balanced_matched_v1),
        "label": 0,
        "description": "Final sampled T2D-negative set, 1:1 length-stratified against positives"
    },
    {
        "dataset": "Protein Classification Dataset Balanced v1",
        "n_rows": len(t2d_protein_classification_dataset_balanced_v1),
        "label": "0/1",
        "description": "Final binary classification dataset for protein-sequence modelling"
    }
])

dataset_v1_summary.to_csv(
    output_dir / "t2d_protein_dataset_v1_summary.csv",
    index=False
)

dataset_v1_summary

,dataset,n_rows,label,description
0,Positive Protein-Ready v1,910,1,T2D-positive reviewed human UniProt protein se...
1,Negative Candidate Pool Clean v1,9459,candidate_pool,Strict negative/background candidate pool befo...
2,Negative Balanced Length-Matched v1,910,0,"Final sampled T2D-negative set, 1:1 length-str..."
3,Protein Classification Dataset Balanced v1,1820,0/1,Final binary classification dataset for protei...


In [107]:
positive_protein_ready_v1["label"].value_counts(dropna=False)

label
1.0    905
NaN      5
Name: count, dtype: int64

In [108]:
positive_protein_ready_v1[
    positive_protein_ready_v1["label"].isna()
][
    [
        "target_id",
        "target_symbol",
        "uniprot_accession",
        "mapping_route",
        "label",
        "label_name",
        "label_source",
        "label_rule",
        "label_version"
    ]
]

,target_id,target_symbol,uniprot_accession,mapping_route,label,label_name,label_source,label_rule,label_version
906,ENSG00000166436,TRIM66,O15016,fallback_exact_primary_gene_symbol,NaN,NaN,NaN,NaN,NaN
907,ENSG00000175164,ABO,P16442,fallback_exact_primary_gene_symbol,NaN,NaN,NaN,NaN,NaN
908,ENSG00000084734,GCKR,Q14397,fallback_exact_primary_gene_symbol,NaN,NaN,NaN,NaN,NaN
909,ENSG00000174326,SLC16A11,Q8NCK7,fallback_exact_primary_gene_symbol,NaN,NaN,NaN,NaN,NaN
910,ENSG00000060749,QSER1,Q2KHR3,fallback_exact_primary_gene_symbol,NaN,NaN,NaN,NaN,NaN


In [109]:
missing_positive_label_mask = positive_protein_ready_v1["label"].isna()

positive_protein_ready_v1.loc[missing_positive_label_mask, "label"] = 1
positive_protein_ready_v1.loc[missing_positive_label_mask, "label_name"] = "T2D_positive"
positive_protein_ready_v1.loc[missing_positive_label_mask, "label_source"] = "Open Targets Platform"
positive_protein_ready_v1.loc[missing_positive_label_mask, "label_rule"] = (
    "OpenTargets_association_score_ge_0.30"
)
positive_protein_ready_v1.loc[missing_positive_label_mask, "label_version"] = (
    "Positive_Label_v1"
)

In [110]:
positive_protein_ready_v1["label"] = positive_protein_ready_v1["label"].astype(int)

In [111]:
positive_label_validation = {
    "n_positive_rows": len(positive_protein_ready_v1),
    "label_1_count": int((positive_protein_ready_v1["label"] == 1).sum()),
    "n_missing_labels": int(positive_protein_ready_v1["label"].isna().sum()),
    "n_fallback_rows": int(
        (positive_protein_ready_v1["mapping_route"] == "fallback_exact_primary_gene_symbol").sum()
    )
}

positive_label_validation

{'n_positive_rows': 910,
 'label_1_count': 910,
 'n_missing_labels': 0,
 'n_fallback_rows': 5}

In [112]:
positive_protein_ready_v1.to_csv(
    output_dir / "t2d_positive_protein_ready_v1.csv",
    index=False
)

In [113]:
positive_model_v1 = positive_protein_ready_v1.copy()

positive_model_v1["gene_id"] = positive_model_v1["target_id"]
positive_model_v1["gene_symbol"] = positive_model_v1["target_symbol"]
positive_model_v1["gene_name"] = positive_model_v1["target_name"]

positive_model_v1["class_source"] = "Category_A_T2D_positive_label"
positive_model_v1["dataset_role"] = "positive"
positive_model_v1["classification_dataset_version"] = (
    "T2D_Protein_Classification_Balanced_v1"
)

In [114]:
negative_model_v1 = negative_balanced_matched_v1.copy()

negative_model_v1["gene_id"] = negative_model_v1["ensembl_gene_id"]
negative_model_v1["gene_symbol"] = negative_model_v1["ensembl_gene_name"]
negative_model_v1["gene_name"] = negative_model_v1["ensembl_gene_name"]

negative_model_v1["class_source"] = (
    "Category_B_negative_clean_pool_lengthmatched_sampling"
)
negative_model_v1["dataset_role"] = "negative"
negative_model_v1["classification_dataset_version"] = (
    "T2D_Protein_Classification_Balanced_v1"
)

In [115]:
for col in model_final_columns:
    if col not in positive_model_v1.columns:
        positive_model_v1[col] = np.nan

for col in model_final_columns:
    if col not in negative_model_v1.columns:
        negative_model_v1[col] = np.nan

t2d_protein_classification_dataset_balanced_v1 = pd.concat(
    [
        positive_model_v1[model_final_columns],
        negative_model_v1[model_final_columns]
    ],
    ignore_index=True
)

t2d_protein_classification_dataset_balanced_v1 = (
    t2d_protein_classification_dataset_balanced_v1
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

C:\Users\khoah\AppData\Local\Temp\ipykernel_21600\738902061.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  t2d_protein_classification_dataset_balanced_v1 = pd.concat(


In [116]:
final_classification_validation = {
    "n_total_rows": len(t2d_protein_classification_dataset_balanced_v1),
    "n_positive_rows": int(
        (t2d_protein_classification_dataset_balanced_v1["label"] == 1).sum()
    ),
    "n_negative_rows": int(
        (t2d_protein_classification_dataset_balanced_v1["label"] == 0).sum()
    ),
    "n_missing_labels": int(
        t2d_protein_classification_dataset_balanced_v1["label"].isna().sum()
    ),
    "n_unique_gene_ids": (
        t2d_protein_classification_dataset_balanced_v1["gene_id"].nunique()
    ),
    "n_unique_uniprot_accessions": (
        t2d_protein_classification_dataset_balanced_v1["uniprot_accession"].nunique()
    ),
    "n_missing_sequences": int(
        t2d_protein_classification_dataset_balanced_v1["sequence"].isna().sum()
    ),
    "n_duplicate_gene_ids": int(
        t2d_protein_classification_dataset_balanced_v1["gene_id"]
        .duplicated()
        .sum()
    ),
    "n_duplicate_uniprot_accessions": int(
        t2d_protein_classification_dataset_balanced_v1["uniprot_accession"]
        .duplicated()
        .sum()
    ),
}

final_classification_validation

{'n_total_rows': 1820,
 'n_positive_rows': 910,
 'n_negative_rows': 910,
 'n_missing_labels': 0,
 'n_unique_gene_ids': 1820,
 'n_unique_uniprot_accessions': 1820,
 'n_missing_sequences': 0,
 'n_duplicate_gene_ids': 0,
 'n_duplicate_uniprot_accessions': 0}

In [117]:
t2d_protein_classification_dataset_balanced_v1.to_csv(
    output_dir / "t2d_protein_classification_dataset_balanced_v1.csv",
    index=False
)

In [118]:
def write_classification_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['gene_id']}|"
                f"{row['gene_symbol']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"label={int(row['label'])}"
            )

            sequence = str(row["sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")


combined_fasta_path = (
    output_dir / "t2d_protein_classification_dataset_balanced_v1.fasta"
)

write_classification_fasta(
    t2d_protein_classification_dataset_balanced_v1,
    combined_fasta_path
)

print("Saved:", combined_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_protein_classification_dataset_balanced_v1.fasta


In [119]:
coordinate_cols = [
    "gene_id",
    "gene_symbol",
    "label",
    "ensembl_gene_id",
    "ensembl_gene_name",
    "gene_type",
    "chromosome_or_scaffold",
    "gene_start_bp",
    "gene_end_bp",
    "strand"
]

t2d_protein_classification_dataset_balanced_v1[
    coordinate_cols
].head()

,gene_id,gene_symbol,label,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,ENSG00000084092,NOA1,0,ENSG00000084092,NOA1,protein_coding,4,56962876.0,56977623.0,-1.0
1,ENSG00000185716,MOSMO,0,ENSG00000185716,MOSMO,protein_coding,16,22007638.0,22087534.0,1.0
2,ENSG00000106113,CRHR2,1,ENSG00000106113,CRHR2,protein_coding,7,30651444.0,30700129.0,-1.0
3,ENSG00000165392,WRN,1,ENSG00000165392,WRN,protein_coding,8,31033779.0,31176138.0,1.0
4,ENSG00000131196,NFATC1,1,ENSG00000131196,NFATC1,protein_coding,18,79395856.0,79529325.0,1.0


In [120]:
coordinate_missing_summary = pd.DataFrame({
    "column": [
        "ensembl_gene_id",
        "chromosome_or_scaffold",
        "gene_start_bp",
        "gene_end_bp",
        "strand"
    ],
    "missing_count": [
        t2d_protein_classification_dataset_balanced_v1["ensembl_gene_id"].isna().sum(),
        t2d_protein_classification_dataset_balanced_v1["chromosome_or_scaffold"].isna().sum(),
        t2d_protein_classification_dataset_balanced_v1["gene_start_bp"].isna().sum(),
        t2d_protein_classification_dataset_balanced_v1["gene_end_bp"].isna().sum(),
        t2d_protein_classification_dataset_balanced_v1["strand"].isna().sum()
    ]
})

coordinate_missing_summary["missing_pct"] = (
    coordinate_missing_summary["missing_count"]
    / len(t2d_protein_classification_dataset_balanced_v1)
    * 100
)

coordinate_missing_summary

,column,missing_count,missing_pct
0,ensembl_gene_id,0,0.0
1,chromosome_or_scaffold,0,0.0
2,gene_start_bp,0,0.0
3,gene_end_bp,0,0.0
4,strand,0,0.0


In [121]:
coordinate_missing_by_label = (
    t2d_protein_classification_dataset_balanced_v1
    .groupby("label")
    .agg(
        n_rows=("gene_id", "size"),
        missing_ensembl_gene_id=("ensembl_gene_id", lambda x: x.isna().sum()),
        missing_chr=("chromosome_or_scaffold", lambda x: x.isna().sum()),
        missing_start=("gene_start_bp", lambda x: x.isna().sum()),
        missing_end=("gene_end_bp", lambda x: x.isna().sum()),
        missing_strand=("strand", lambda x: x.isna().sum())
    )
    .reset_index()
)

coordinate_missing_by_label

,label,n_rows,missing_ensembl_gene_id,missing_chr,missing_start,missing_end,missing_strand
0,0,910,0,0,0,0,0
1,1,910,0,0,0,0,0


In [122]:
chromosome_distribution = (
    t2d_protein_classification_dataset_balanced_v1[
        "chromosome_or_scaffold"
    ]
    .value_counts(dropna=False)
    .reset_index()
)

chromosome_distribution.columns = [
    "chromosome_or_scaffold",
    "n_genes"
]

chromosome_distribution.head(40)

,chromosome_or_scaffold,n_genes
0,1,153
1,11,118
2,2,114
3,19,102
4,3,101
5,17,94
6,7,93
7,6,93
8,5,92
9,12,88


In [123]:
invalid_coordinate_rows = t2d_protein_classification_dataset_balanced_v1[
    (
        t2d_protein_classification_dataset_balanced_v1["gene_start_bp"].notna()
    )
    &
    (
        t2d_protein_classification_dataset_balanced_v1["gene_end_bp"].notna()
    )
    &
    (
        t2d_protein_classification_dataset_balanced_v1["gene_start_bp"]
        >
        t2d_protein_classification_dataset_balanced_v1["gene_end_bp"]
    )
]

print("Invalid coordinate rows:", invalid_coordinate_rows.shape)

Invalid coordinate rows: (0, 30)


In [124]:
primary_chromosomes = {
    "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
    "11", "12", "13", "14", "15", "16", "17", "18", "19",
    "20", "21", "22", "X", "Y", "MT"
}

In [125]:
final_gene_ids = set(
    t2d_protein_classification_dataset_balanced_v1["ensembl_gene_id"]
)

df_biomart_final_genes_all_rows = df_biomart_clean[
    df_biomart_clean["ensembl_gene_id"].isin(final_gene_ids)
].copy()

df_biomart_final_genes_all_rows["is_primary_chromosome"] = (
    df_biomart_final_genes_all_rows["chromosome_or_scaffold"]
    .astype(str)
    .isin(primary_chromosomes)
)

print("BioMart rows for final 1,820 genes:",
      df_biomart_final_genes_all_rows.shape)

print("Unique ENSG:",
      df_biomart_final_genes_all_rows["ensembl_gene_id"].nunique())

BioMart rows for final 1,820 genes: (30937, 12)
Unique ENSG: 1820


In [126]:
gene_locus_summary = (
    df_biomart_final_genes_all_rows
    .groupby("ensembl_gene_id")
    .agg(
        n_unique_locations=("chromosome_or_scaffold", "nunique"),
        has_primary_location=("is_primary_chromosome", "max"),
        all_locations=("chromosome_or_scaffold", lambda x: sorted(set(map(str, x))))
    )
    .reset_index()
)

gene_locus_summary["location_status"] = np.select(
    [
        (gene_locus_summary["n_unique_locations"] == 1) &
        (gene_locus_summary["has_primary_location"] == True),

        (gene_locus_summary["n_unique_locations"] > 1) &
        (gene_locus_summary["has_primary_location"] == True),

        (gene_locus_summary["has_primary_location"] == False)
    ],
    [
        "single_primary_location",
        "multiple_locations_with_primary_available",
        "non_primary_only"
    ],
    default="other"
)

gene_locus_summary["location_status"].value_counts()

location_status
single_primary_location    1776
non_primary_only             44
Name: count, dtype: int64

In [127]:
current_non_primary_rows = (
    t2d_protein_classification_dataset_balanced_v1[
        ~t2d_protein_classification_dataset_balanced_v1[
            "chromosome_or_scaffold"
        ].astype(str).isin(primary_chromosomes)
    ][
        [
            "gene_id",
            "gene_symbol",
            "label",
            "ensembl_gene_id",
            "chromosome_or_scaffold",
            "gene_start_bp",
            "gene_end_bp",
            "strand"
        ]
    ]
    .copy()
)

print("Current non-primary coordinate rows:",
      current_non_primary_rows.shape)

current_non_primary_rows.head(50)

Current non-primary coordinate rows: (44, 8)


,gene_id,gene_symbol,label,ensembl_gene_id,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
5,ENSG00000276469,GPR179,0,ENSG00000276469,HSCHR17_7_CTG4,2360558.0,2378758.0,-1.0
56,ENSG00000285049,CPSF1,0,ENSG00000285049,HG2419_PATCH,123832.0,139938.0,-1.0
94,ENSG00000282841,CTAGE1,0,ENSG00000282841,HSCHR18_5_CTG1_1,24920.0,29236.0,-1.0
107,ENSG00000283847,HBP1,0,ENSG00000283847,HG2266_PATCH,14483.0,48044.0,1.0
134,ENSG00000278613,NAIP,0,ENSG00000278613,HSCHR5_2_CTG1_1,385732.0,442341.0,1.0
167,ENSG00000262243,CES1,0,ENSG00000262243,HSCHR16_1_CTG3_1,26724.0,57055.0,-1.0
224,ENSG00000278488,NAPRT,0,ENSG00000278488,HSCHR8_3_CTG7,64825.0,68689.0,-1.0
228,ENSG00000288475,CASP8AP2,0,ENSG00000288475,HG2121_PATCH,25225.0,72124.0,1.0
279,ENSG00000285501,AGBL2,0,ENSG00000285501,HG2114_PATCH,91958.0,147756.0,-1.0
324,ENSG00000277104,TADA2A,0,ENSG00000277104,HSCHR17_7_CTG4,1646030.0,1718900.0,1.0


In [128]:
current_non_primary_with_primary_check = current_non_primary_rows.merge(
    gene_locus_summary[
        [
            "ensembl_gene_id",
            "n_unique_locations",
            "has_primary_location",
            "all_locations",
            "location_status"
        ]
    ],
    on="ensembl_gene_id",
    how="left"
)

current_non_primary_with_primary_check[
    "location_status"
].value_counts()

location_status
non_primary_only    44
Name: count, dtype: int64

In [129]:
primary_chromosomes = {
    "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
    "11", "12", "13", "14", "15", "16", "17", "18", "19",
    "20", "21", "22", "X", "Y", "MT"
}

negative_non_primary_only_to_remove = negative_balanced_matched_v1[
    ~negative_balanced_matched_v1["chromosome_or_scaffold"]
    .astype(str)
    .isin(primary_chromosomes)
].copy()

print("Negative non-primary-only rows to remove:",
      negative_non_primary_only_to_remove.shape)

negative_non_primary_only_to_remove[
    [
        "ensembl_gene_id",
        "ensembl_gene_name",
        "uniprot_accession",
        "sequence_length_fasta",
        "length_bin",
        "chromosome_or_scaffold"
    ]
].head()

Negative non-primary-only rows to remove: (44, 25)


,ensembl_gene_id,ensembl_gene_name,uniprot_accession,sequence_length_fasta,length_bin,chromosome_or_scaffold
24,ENSG00000275932,DUSP14,O95147,198,"(40.999, 210.0]",HSCHR17_7_CTG4
87,ENSG00000273607,CHCHD10,Q8WYQ3,142,"(40.999, 210.0]",HSCHR22_1_CTG7
110,ENSG00000283903,FGF11,Q92914,225,"(210.0, 315.0]",HG2046_PATCH
155,ENSG00000283979,RFLNB,Q8N5W9,214,"(210.0, 315.0]",HG2285_HG106_HG2252_PATCH
187,ENSG00000277273,CDK7,P50613,346,"(315.0, 398.0]",HSCHR5_2_CTG1_1


In [130]:
bins_to_replace_summary = (
    negative_non_primary_only_to_remove["length_bin"]
    .value_counts()
    .sort_index()
)

bins_to_replace_summary

length_bin
(40.999, 210.0]      2
(210.0, 315.0]       2
(315.0, 398.0]       2
(398.0, 469.0]       6
(469.0, 561.0]       5
(561.0, 683.2]       3
(683.2, 852.6]       3
(852.6, 1078.0]      6
(1078.0, 1447.3]     7
(1447.3, 34350.0]    8
Name: count, dtype: int64

In [131]:
already_selected_negative_uniprot = set(
    negative_balanced_matched_v1["uniprot_accession"]
)

negative_replacement_candidate_pool = negative_candidate_pool_clean_v1[
    negative_candidate_pool_clean_v1["chromosome_or_scaffold"]
    .astype(str)
    .isin(primary_chromosomes)
].copy()

negative_replacement_candidate_pool = negative_replacement_candidate_pool[
    ~negative_replacement_candidate_pool["uniprot_accession"]
    .isin(already_selected_negative_uniprot)
].copy()

print("Replacement candidate pool shape:",
      negative_replacement_candidate_pool.shape)
print("Unique ENSG:",
      negative_replacement_candidate_pool["ensembl_gene_id"].nunique())
print("Unique UniProt:",
      negative_replacement_candidate_pool["uniprot_accession"].nunique())

Replacement candidate pool shape: (8206, 22)
Unique ENSG: 8206
Unique UniProt: 8206


In [132]:
negative_replacement_candidate_pool["length_bin"] = pd.cut(
    negative_replacement_candidate_pool["sequence_length_fasta"],
    bins=bin_edges,
    include_lowest=True
)

In [133]:
replacement_availability = pd.DataFrame({
    "n_needed_to_replace": bins_to_replace_summary,
    "n_candidates_available": (
        negative_replacement_candidate_pool["length_bin"]
        .value_counts()
        .sort_index()
    )
}).fillna(0)

replacement_availability

,n_needed_to_replace,n_candidates_available
length_bin,,
"(40.999, 210.0]",2,1537
"(210.0, 315.0]",2,1509
"(315.0, 398.0]",2,1081
"(398.0, 469.0]",6,706
"(469.0, 561.0]",5,834
"(561.0, 683.2]",3,744
"(683.2, 852.6]",3,677
"(852.6, 1078.0]",6,474
"(1078.0, 1447.3]",7,317


In [134]:
replacement_parts = []

for bin_label, n_needed in bins_to_replace_summary.items():
    candidates_in_bin = negative_replacement_candidate_pool[
        negative_replacement_candidate_pool["length_bin"] == bin_label
    ]

    sampled_replacements = candidates_in_bin.sample(
        n=n_needed,
        random_state=42
    )

    replacement_parts.append(sampled_replacements)

negative_primary_replacements_v1 = pd.concat(
    replacement_parts,
    ignore_index=True
)

print("Replacement negatives shape:",
      negative_primary_replacements_v1.shape)
print("Unique ENSG:",
      negative_primary_replacements_v1["ensembl_gene_id"].nunique())
print("Unique UniProt:",
      negative_primary_replacements_v1["uniprot_accession"].nunique())

Replacement negatives shape: (44, 23)
Unique ENSG: 44
Unique UniProt: 44


In [135]:
negative_balanced_primary_only_v1 = negative_balanced_matched_v1[
    negative_balanced_matched_v1["chromosome_or_scaffold"]
    .astype(str)
    .isin(primary_chromosomes)
].copy()

negative_balanced_primary_only_v1 = pd.concat(
    [
        negative_balanced_primary_only_v1,
        negative_primary_replacements_v1
    ],
    ignore_index=True
)

print("Final negative primary-only set shape:",
      negative_balanced_primary_only_v1.shape)
print("Unique ENSG:",
      negative_balanced_primary_only_v1["ensembl_gene_id"].nunique())
print("Unique UniProt:",
      negative_balanced_primary_only_v1["uniprot_accession"].nunique())

Final negative primary-only set shape: (910, 25)
Unique ENSG: 910
Unique UniProt: 910


In [136]:
primary_only_length_bin_check = pd.DataFrame({
    "positive_count": (
        positive_for_sampling["length_bin"]
        .value_counts()
        .sort_index()
    ),
    "negative_primary_only_count": (
        negative_balanced_primary_only_v1["length_bin"]
        .value_counts()
        .sort_index()
    )
})

primary_only_length_bin_check

,positive_count,negative_primary_only_count
length_bin,,
"(40.999, 210.0]",92,92
"(210.0, 315.0]",91,91
"(315.0, 398.0]",92,92
"(398.0, 469.0]",91,91
"(469.0, 561.0]",90,90
"(561.0, 683.2]",90,90
"(683.2, 852.6]",91,91
"(852.6, 1078.0]",92,92
"(1078.0, 1447.3]",90,90


In [137]:
negative_primary_only_validation = {
    "n_rows": len(negative_balanced_primary_only_v1),
    "n_unique_ENSG": negative_balanced_primary_only_v1["ensembl_gene_id"].nunique(),
    "n_unique_uniprot": negative_balanced_primary_only_v1["uniprot_accession"].nunique(),
    "n_non_primary_rows_remaining": int(
        ~negative_balanced_primary_only_v1["chromosome_or_scaffold"]
        .astype(str)
        .isin(primary_chromosomes)
    ).sum() if False else int(
        (
            ~negative_balanced_primary_only_v1["chromosome_or_scaffold"]
            .astype(str)
            .isin(primary_chromosomes)
        ).sum()
    ),
    "n_overlap_positive_ENSG": int(
        negative_balanced_primary_only_v1["ensembl_gene_id"]
        .isin(positive_protein_ready_v1["target_id"])
        .sum()
    ),
    "n_overlap_positive_uniprot": int(
        negative_balanced_primary_only_v1["uniprot_accession"]
        .isin(positive_protein_ready_v1["uniprot_accession"])
        .sum()
    )
}

negative_primary_only_validation

{'n_rows': 910,
 'n_unique_ENSG': 910,
 'n_unique_uniprot': 910,
 'n_non_primary_rows_remaining': 0,
 'n_overlap_positive_ENSG': 0,
 'n_overlap_positive_uniprot': 0}

In [138]:
negative_balanced_primary_only_v1["label"] = 0
negative_balanced_primary_only_v1["label_name"] = "T2D_negative"

negative_balanced_primary_only_v1["negative_sample_version"] = (
    "Negative_Balanced_LengthMatched_PrimaryCoordinates_v1"
)

negative_balanced_primary_only_v1["negative_sampling_rule"] = (
    "Sampled from Negative Candidate Pool Clean v1 using 1:1 "
    "positive-negative balance with sequence-length-stratified sampling "
    "based on positive-set decile bins; non-primary patch/scaffold-only "
    "genes were excluded and replaced within the same length bins; random_state=42"
)

In [139]:
negative_balanced_primary_only_v1.to_csv(
    output_dir / "t2d_negative_balanced_lengthmatched_primary_only_v1.csv",
    index=False
)

print(
    "Saved:",
    output_dir / "t2d_negative_balanced_lengthmatched_primary_only_v1.csv"
)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_balanced_lengthmatched_primary_only_v1.csv


In [140]:
def write_negative_primary_only_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['ensembl_gene_id']}|"
                f"{row['ensembl_gene_name']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"label=0"
            )

            sequence = str(row["sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")


negative_primary_only_fasta_path = (
    output_dir / "t2d_negative_balanced_lengthmatched_primary_only_v1.fasta"
)

write_negative_primary_only_fasta(
    negative_balanced_primary_only_v1,
    negative_primary_only_fasta_path
)

print("Saved:", negative_primary_only_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_negative_balanced_lengthmatched_primary_only_v1.fasta


In [141]:
positive_model_v1 = positive_protein_ready_v1.copy()

positive_model_v1["gene_id"] = positive_model_v1["target_id"]
positive_model_v1["gene_symbol"] = positive_model_v1["target_symbol"]
positive_model_v1["gene_name"] = positive_model_v1["target_name"]

positive_model_v1["class_source"] = "Category_A_T2D_positive_label"
positive_model_v1["dataset_role"] = "positive"
positive_model_v1["classification_dataset_version"] = (
    "T2D_Protein_Classification_Balanced_PrimaryCoordinates_v1"
)

In [142]:
negative_model_primary_v1 = negative_balanced_primary_only_v1.copy()

negative_model_primary_v1["gene_id"] = (
    negative_model_primary_v1["ensembl_gene_id"]
)
negative_model_primary_v1["gene_symbol"] = (
    negative_model_primary_v1["ensembl_gene_name"]
)
negative_model_primary_v1["gene_name"] = (
    negative_model_primary_v1["ensembl_gene_name"]
)

negative_model_primary_v1["class_source"] = (
    "Category_B_negative_clean_pool_lengthmatched_primary_coordinates"
)
negative_model_primary_v1["dataset_role"] = "negative"
negative_model_primary_v1["classification_dataset_version"] = (
    "T2D_Protein_Classification_Balanced_PrimaryCoordinates_v1"
)

In [143]:
model_final_columns = [
    # Core identity
    "gene_id",
    "gene_symbol",
    "gene_name",
    "uniprot_accession",
    "uniprot_entry_name",

    # Protein input
    "sequence",
    "sequence_length_fasta",

    # Label
    "label",
    "label_name",
    "dataset_role",
    "class_source",
    "classification_dataset_version",

    # Positive-only evidence fields
    "association_score",
    "positive_tier",
    "label_source",
    "label_rule",
    "label_version",

    # Negative-only fields
    "negative_sample_version",
    "negative_sampling_rule",

    # Sequence/mapping audit
    "mapping_route",
    "uniprot_to_ensg_status",
    "representative_sequence_rule",
    "length_bin",

    # Genomic / mapping metadata
    "ensembl_gene_id",
    "ensembl_gene_name",
    "gene_type",
    "chromosome_or_scaffold",
    "gene_start_bp",
    "gene_end_bp",
    "strand"
]

In [144]:
for col in model_final_columns:
    if col not in positive_model_v1.columns:
        positive_model_v1[col] = np.nan

for col in model_final_columns:
    if col not in negative_model_primary_v1.columns:
        negative_model_primary_v1[col] = np.nan

In [145]:
t2d_protein_classification_dataset_balanced_primary_only_v1 = pd.concat(
    [
        positive_model_v1[model_final_columns],
        negative_model_primary_v1[model_final_columns]
    ],
    ignore_index=True
)

t2d_protein_classification_dataset_balanced_primary_only_v1 = (
    t2d_protein_classification_dataset_balanced_primary_only_v1
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(
    "Final primary-only classification dataset shape:",
    t2d_protein_classification_dataset_balanced_primary_only_v1.shape
)

t2d_protein_classification_dataset_balanced_primary_only_v1[
    "label"
].value_counts()

Final primary-only classification dataset shape: (1820, 30)


C:\Users\khoah\AppData\Local\Temp\ipykernel_21600\2908491640.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  t2d_protein_classification_dataset_balanced_primary_only_v1 = pd.concat(


label
0    910
1    910
Name: count, dtype: int64

In [146]:
final_primary_only_classification_validation = {
    "n_total_rows": len(
        t2d_protein_classification_dataset_balanced_primary_only_v1
    ),
    "n_positive_rows": int(
        (
            t2d_protein_classification_dataset_balanced_primary_only_v1["label"]
            == 1
        ).sum()
    ),
    "n_negative_rows": int(
        (
            t2d_protein_classification_dataset_balanced_primary_only_v1["label"]
            == 0
        ).sum()
    ),
    "n_missing_labels": int(
        t2d_protein_classification_dataset_balanced_primary_only_v1[
            "label"
        ].isna().sum()
    ),
    "n_unique_gene_ids": (
        t2d_protein_classification_dataset_balanced_primary_only_v1[
            "gene_id"
        ].nunique()
    ),
    "n_unique_uniprot_accessions": (
        t2d_protein_classification_dataset_balanced_primary_only_v1[
            "uniprot_accession"
        ].nunique()
    ),
    "n_missing_sequences": int(
        t2d_protein_classification_dataset_balanced_primary_only_v1[
            "sequence"
        ].isna().sum()
    ),
    "n_duplicate_gene_ids": int(
        t2d_protein_classification_dataset_balanced_primary_only_v1[
            "gene_id"
        ].duplicated().sum()
    ),
    "n_duplicate_uniprot_accessions": int(
        t2d_protein_classification_dataset_balanced_primary_only_v1[
            "uniprot_accession"
        ].duplicated().sum()
    ),
    "n_non_primary_coordinate_rows": int(
        (
            ~t2d_protein_classification_dataset_balanced_primary_only_v1[
                "chromosome_or_scaffold"
            ].astype(str).isin(primary_chromosomes)
        ).sum()
    )
}

final_primary_only_classification_validation

{'n_total_rows': 1820,
 'n_positive_rows': 910,
 'n_negative_rows': 910,
 'n_missing_labels': 0,
 'n_unique_gene_ids': 1820,
 'n_unique_uniprot_accessions': 1820,
 'n_missing_sequences': 0,
 'n_duplicate_gene_ids': 0,
 'n_duplicate_uniprot_accessions': 0,
 'n_non_primary_coordinate_rows': 0}

In [147]:
t2d_protein_classification_dataset_balanced_primary_only_v1.to_csv(
    output_dir / "t2d_protein_classification_dataset_balanced_primary_only_v1.csv",
    index=False
)

print(
    "Saved:",
    output_dir / "t2d_protein_classification_dataset_balanced_primary_only_v1.csv"
)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_protein_classification_dataset_balanced_primary_only_v1.csv


In [148]:
def write_classification_primary_only_fasta(df, fasta_path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['gene_id']}|"
                f"{row['gene_symbol']}|"
                f"{row['uniprot_accession']}|"
                f"{row['uniprot_entry_name']}|"
                f"label={int(row['label'])}"
            )

            sequence = str(row["sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")


combined_primary_only_fasta_path = (
    output_dir / "t2d_protein_classification_dataset_balanced_primary_only_v1.fasta"
)

write_classification_primary_only_fasta(
    t2d_protein_classification_dataset_balanced_primary_only_v1,
    combined_primary_only_fasta_path
)

print("Saved:", combined_primary_only_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_protein_classification_dataset_balanced_primary_only_v1.fasta


In [149]:
gene_coordinate_master_v1 = (
    t2d_protein_classification_dataset_balanced_primary_only_v1[
        [
            "gene_id",
            "gene_symbol",
            "gene_name",
            "label",
            "dataset_role",
            "ensembl_gene_id",
            "ensembl_gene_name",
            "gene_type",
            "chromosome_or_scaffold",
            "gene_start_bp",
            "gene_end_bp",
            "strand",
            "uniprot_accession",
            "uniprot_entry_name",
            "sequence_length_fasta"
        ]
    ]
    .copy()
)

gene_coordinate_master_v1["coordinate_source"] = "Ensembl BioMart"
gene_coordinate_master_v1["coordinate_scope"] = (
    "Primary chromosomes only: 1-22, X, Y, MT"
)
gene_coordinate_master_v1["coordinate_master_version"] = (
    "Gene_Coordinate_Master_v1"
)

print("Gene coordinate master shape:", gene_coordinate_master_v1.shape)
gene_coordinate_master_v1.head()

Gene coordinate master shape: (1820, 18)


,gene_id,gene_symbol,gene_name,label,dataset_role,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,uniprot_accession,uniprot_entry_name,sequence_length_fasta,coordinate_source,coordinate_scope,coordinate_master_version
0,ENSG00000161618,ALDH16A1,ALDH16A1,0,negative,ENSG00000161618,ALDH16A1,protein_coding,19,49453146.0,49471052.0,1.0,Q8IZ83,A16A1_HUMAN,802.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1
1,ENSG00000224420,ADM5,ADM5,0,negative,ENSG00000224420,ADM5,protein_coding,19,49688953.0,49690575.0,1.0,C9JUS6,ADM5_HUMAN,153.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1
2,ENSG00000106113,CRHR2,corticotropin releasing hormone receptor 2,1,positive,ENSG00000106113,CRHR2,protein_coding,7,30651444.0,30700129.0,-1.0,Q13324,CRFR2_HUMAN,411.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1
3,ENSG00000165392,WRN,WRN RecQ like helicase,1,positive,ENSG00000165392,WRN,protein_coding,8,31033779.0,31176138.0,1.0,Q14191,WRN_HUMAN,1432.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1
4,ENSG00000131196,NFATC1,nuclear factor of activated T cells 1,1,positive,ENSG00000131196,NFATC1,protein_coding,18,79395856.0,79529325.0,1.0,O95644,NFAC1_HUMAN,943.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1


In [150]:
gene_coordinate_master_validation = {
    "n_rows": len(gene_coordinate_master_v1),
    "n_unique_gene_ids": gene_coordinate_master_v1["gene_id"].nunique(),
    "n_positive_rows": int((gene_coordinate_master_v1["label"] == 1).sum()),
    "n_negative_rows": int((gene_coordinate_master_v1["label"] == 0).sum()),
    "n_missing_coordinates": int(
        gene_coordinate_master_v1[
            [
                "chromosome_or_scaffold",
                "gene_start_bp",
                "gene_end_bp",
                "strand"
            ]
        ].isna().any(axis=1).sum()
    ),
    "n_non_primary_coordinate_rows": int(
        (
            ~gene_coordinate_master_v1["chromosome_or_scaffold"]
            .astype(str)
            .isin(primary_chromosomes)
        ).sum()
    ),
    "n_invalid_start_gt_end": int(
        (
            gene_coordinate_master_v1["gene_start_bp"]
            >
            gene_coordinate_master_v1["gene_end_bp"]
        ).sum()
    )
}

gene_coordinate_master_validation

{'n_rows': 1820,
 'n_unique_gene_ids': 1820,
 'n_positive_rows': 910,
 'n_negative_rows': 910,
 'n_missing_coordinates': 0,
 'n_non_primary_coordinate_rows': 0,
 'n_invalid_start_gt_end': 0}

In [151]:
gene_coordinate_master_v1.to_csv(
    output_dir / "t2d_gene_coordinate_master_primary_only_v1.csv",
    index=False
)

print(
    "Saved:",
    output_dir / "t2d_gene_coordinate_master_primary_only_v1.csv"
)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_gene_coordinate_master_primary_only_v1.csv


In [152]:
# Reference genome FASTA
genome_fasta_path = Path(
    r"D:\khoane\MAHE\Project\dataset\UniPro\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
)

# Output folder for Category D
category_d_output_dir = Path(
    r"D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output"
)

category_d_output_dir.mkdir(parents=True, exist_ok=True)

print("Genome FASTA exists:", genome_fasta_path.exists())
print("Output folder:", category_d_output_dir)

Genome FASTA exists: True
Output folder: D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output


In [154]:
from pyfaidx import Fasta
genome = Fasta(
    str(genome_fasta_path),
    as_raw=True,
    sequence_always_upper=True
)

In [156]:
fasta_chromosome_keys = list(genome.keys())

print("Number of FASTA records:", len(fasta_chromosome_keys))
print("First 30 FASTA keys:")
print(fasta_chromosome_keys[:30])

Number of FASTA records: 194
First 30 FASTA keys:
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9', 'MT', 'X', 'Y', 'KI270728.1', 'KI270727.1', 'KI270442.1', 'KI270729.1', 'GL000225.1']


In [157]:
coordinate_chromosomes = set(
    gene_coordinate_master_v1["chromosome_or_scaffold"].astype(str)
)

fasta_chromosomes = set(map(str, fasta_chromosome_keys))

missing_chromosomes_in_fasta = sorted(
    coordinate_chromosomes - fasta_chromosomes
)

print("Chromosomes used by coordinate master:")
print(sorted(coordinate_chromosomes))

print("\nChromosomes missing from FASTA:")
print(missing_chromosomes_in_fasta)

Chromosomes used by coordinate master:
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9', 'MT', 'X', 'Y']

Chromosomes missing from FASTA:
[]


In [158]:
DNA_COMPLEMENT_TABLE = str.maketrans({
    "A": "T",
    "T": "A",
    "C": "G",
    "G": "C",
    "N": "N"
})

def reverse_complement_dna(sequence: str) -> str:
    """
    Return reverse-complement DNA sequence.
    Expected uppercase A/C/G/T/N sequence.
    """
    return sequence.translate(DNA_COMPLEMENT_TABLE)[::-1]

In [159]:
def extract_gene_genomic_sequence(
    chromosome: str,
    gene_start_bp: int,
    gene_end_bp: int,
    strand: int,
    genome_fasta: Fasta
) -> dict:
    """
    Extract whole-gene genomic DNA sequence from Ensembl-style coordinates.

    Ensembl coordinates:
        - 1-based
        - inclusive start and end

    pyfaidx/Python slicing:
        - 0-based start
        - end-exclusive

    Therefore:
        fasta_start = gene_start_bp - 1
        fasta_end   = gene_end_bp
    """

    chromosome = str(chromosome)
    gene_start_bp = int(gene_start_bp)
    gene_end_bp = int(gene_end_bp)
    strand = int(strand)

    expected_coordinate_length = gene_end_bp - gene_start_bp + 1

    # Convert Ensembl 1-based inclusive coordinates to Python-style slicing
    fasta_start = gene_start_bp - 1
    fasta_end = gene_end_bp

    # Extract reference-orientation genomic sequence
    reference_sequence = str(
        genome_fasta[chromosome][fasta_start:fasta_end]
    ).upper()

    if strand == 1:
        oriented_sequence = reference_sequence
        sequence_orientation = "forward_gene_orientation"
    elif strand == -1:
        oriented_sequence = reverse_complement_dna(reference_sequence)
        sequence_orientation = "reverse_complement_gene_orientation"
    else:
        raise ValueError(f"Unexpected strand value: {strand}")

    genomic_sequence_length = len(oriented_sequence)

    return {
        "genomic_sequence": oriented_sequence,
        "genomic_sequence_length": genomic_sequence_length,
        "expected_coordinate_length": expected_coordinate_length,
        "length_match_check": genomic_sequence_length == expected_coordinate_length,
        "sequence_orientation": sequence_orientation,
    }

In [160]:
genomic_extraction_records = []

for idx, row in gene_coordinate_master_v1.iterrows():
    extracted = extract_gene_genomic_sequence(
        chromosome=row["chromosome_or_scaffold"],
        gene_start_bp=row["gene_start_bp"],
        gene_end_bp=row["gene_end_bp"],
        strand=row["strand"],
        genome_fasta=genome
    )

    genomic_extraction_records.append(extracted)

df_genomic_extraction = pd.DataFrame(genomic_extraction_records)

print("Extraction table shape:", df_genomic_extraction.shape)
df_genomic_extraction.head()

Extraction table shape: (1820, 5)


,genomic_sequence,genomic_sequence_length,expected_coordinate_length,length_match_check,sequence_orientation
0,AGGCGGGGCCATCGGGTCCTAAGGAAAAAGTGTGGGCGTGGCTGCT...,17907,17907,True,forward_gene_orientation
1,TTCCTCCCCTAGAATGGCTTGGGGCTTAGGGCTGGGGACTTGCCCT...,1623,1623,True,forward_gene_orientation
2,GCCTCCCCTCCAGGGCCGGGTGGGGTGGGGCTGGCCAGGGTGTGAC...,48686,48686,True,reverse_complement_gene_orientation
3,GGGCGGGTCGGGTAGGTCTCCCGGAGCTGATGTGTACTGTGTGCGC...,142360,142360,True,forward_gene_orientation
4,GAGCCGCCGGGCCGGCGGGGAGGCGGGGGAGGTGTTTTCCAGCTTT...,133470,133470,True,forward_gene_orientation


In [161]:
t2d_whole_gene_genomic_sequences_primary_only_v1 = pd.concat(
    [
        gene_coordinate_master_v1.reset_index(drop=True),
        df_genomic_extraction.reset_index(drop=True)
    ],
    axis=1
)

t2d_whole_gene_genomic_sequences_primary_only_v1[
    [
        "gene_id",
        "gene_symbol",
        "label",
        "chromosome_or_scaffold",
        "gene_start_bp",
        "gene_end_bp",
        "strand",
        "genomic_sequence_length",
        "expected_coordinate_length",
        "length_match_check",
        "sequence_orientation"
    ]
].head()

,gene_id,gene_symbol,label,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,genomic_sequence_length,expected_coordinate_length,length_match_check,sequence_orientation
0,ENSG00000161618,ALDH16A1,0,19,49453146.0,49471052.0,1.0,17907,17907,True,forward_gene_orientation
1,ENSG00000224420,ADM5,0,19,49688953.0,49690575.0,1.0,1623,1623,True,forward_gene_orientation
2,ENSG00000106113,CRHR2,1,7,30651444.0,30700129.0,-1.0,48686,48686,True,reverse_complement_gene_orientation
3,ENSG00000165392,WRN,1,8,31033779.0,31176138.0,1.0,142360,142360,True,forward_gene_orientation
4,ENSG00000131196,NFATC1,1,18,79395856.0,79529325.0,1.0,133470,133470,True,forward_gene_orientation


In [162]:
t2d_whole_gene_genomic_sequences_primary_only_v1["genomic_sequence_source"] = (
    "Ensembl Homo sapiens GRCh38 primary assembly FASTA"
)

t2d_whole_gene_genomic_sequences_primary_only_v1["genomic_sequence_scope"] = (
    "Whole-gene genomic span: gene_start_bp to gene_end_bp inclusive"
)

t2d_whole_gene_genomic_sequences_primary_only_v1["genomic_sequence_version"] = (
    "Whole_Gene_Genomic_Sequence_PrimaryOnly_v1"
)

In [163]:
allowed_dna_characters = set("ACGTN")

invalid_sequence_character_rows = (
    t2d_whole_gene_genomic_sequences_primary_only_v1[
        "genomic_sequence"
    ]
    .apply(lambda seq: not set(seq).issubset(allowed_dna_characters))
)

whole_gene_genomic_validation = {
    "n_total_rows": len(t2d_whole_gene_genomic_sequences_primary_only_v1),
    "n_positive_rows": int(
        (t2d_whole_gene_genomic_sequences_primary_only_v1["label"] == 1).sum()
    ),
    "n_negative_rows": int(
        (t2d_whole_gene_genomic_sequences_primary_only_v1["label"] == 0).sum()
    ),
    "n_unique_gene_ids": (
        t2d_whole_gene_genomic_sequences_primary_only_v1["gene_id"].nunique()
    ),
    "n_missing_genomic_sequences": int(
        t2d_whole_gene_genomic_sequences_primary_only_v1[
            "genomic_sequence"
        ].isna().sum()
    ),
    "n_length_mismatch_rows": int(
        (~t2d_whole_gene_genomic_sequences_primary_only_v1[
            "length_match_check"
        ]).sum()
    ),
    "n_invalid_sequence_character_rows": int(
        invalid_sequence_character_rows.sum()
    ),
    "n_forward_oriented_sequences": int(
        (
            t2d_whole_gene_genomic_sequences_primary_only_v1[
                "sequence_orientation"
            ] == "forward_gene_orientation"
        ).sum()
    ),
    "n_reverse_complement_sequences": int(
        (
            t2d_whole_gene_genomic_sequences_primary_only_v1[
                "sequence_orientation"
            ] == "reverse_complement_gene_orientation"
        ).sum()
    )
}

whole_gene_genomic_validation

{'n_total_rows': 1820,
 'n_positive_rows': 910,
 'n_negative_rows': 910,
 'n_unique_gene_ids': 1820,
 'n_missing_genomic_sequences': 0,
 'n_length_mismatch_rows': 0,
 'n_invalid_sequence_character_rows': 0,
 'n_forward_oriented_sequences': 924,
 'n_reverse_complement_sequences': 896}

In [164]:
t2d_whole_gene_genomic_sequences_primary_only_v1[
    "genomic_sequence_length"
].describe()

count    1.820000e+03
mean     1.141187e+05
std      2.076468e+05
min      2.580000e+02
25%      1.609450e+04
50%      4.710250e+04
75%      1.219900e+05
max      2.473539e+06
Name: genomic_sequence_length, dtype: float64

In [165]:
genomic_length_by_label = (
    t2d_whole_gene_genomic_sequences_primary_only_v1
    .groupby("label")["genomic_sequence_length"]
    .describe()
)

genomic_length_by_label

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,910.0,74453.659341,132384.479700,258.0,12650.75,32467.0,76625.0,1220167.0
1,910.0,153783.702198,256136.879035,297.0,23136.75,69773.0,162891.0,2473539.0


In [166]:
t2d_whole_gene_genomic_sequences_primary_only_v1[
    [
        "gene_id",
        "gene_symbol",
        "label",
        "chromosome_or_scaffold",
        "gene_start_bp",
        "gene_end_bp",
        "genomic_sequence_length"
    ]
].sort_values(
    "genomic_sequence_length",
    ascending=False
).head(20)

,gene_id,gene_symbol,label,chromosome_or_scaffold,gene_start_bp,gene_end_bp,genomic_sequence_length
1775,ENSG00000078328,RBFOX1,1,16,5239802.0,7713340.0,2473539
1670,ENSG00000153707,PTPRD,1,9,8314246.0,10613002.0,2298757
1397,ENSG00000198947,DMD,1,X,31097677.0,33339609.0,2241933
847,ENSG00000185008,ROBO2,1,3,75906695.0,77649964.0,1743270
1660,ENSG00000021645,NRXN3,1,14,78170373.0,79868291.0,1697919
1286,ENSG00000189283,FHIT,1,3,59747277.0,61251459.0,1504183
792,ENSG00000184305,CCSER1,1,4,90127394.0,91605295.0,1477902
1035,ENSG00000179399,GPC5,1,13,91398621.0,92873682.0,1475062
1039,ENSG00000187391,MAGI2,1,7,78017055.0,79453667.0,1436613
863,ENSG00000185345,PRKN,1,6,161347417.0,162727775.0,1380359


In [167]:
whole_gene_genomic_csv_path = (
    category_d_output_dir
    / "t2d_whole_gene_genomic_sequences_primary_only_v1.csv"
)

t2d_whole_gene_genomic_sequences_primary_only_v1.to_csv(
    whole_gene_genomic_csv_path,
    index=False
)

print("Saved:", whole_gene_genomic_csv_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output\t2d_whole_gene_genomic_sequences_primary_only_v1.csv


In [168]:
def write_whole_gene_genomic_fasta(df: pd.DataFrame, fasta_path: Path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['gene_id']}|"
                f"{row['gene_symbol']}|"
                f"label={int(row['label'])}|"
                f"chr={row['chromosome_or_scaffold']}|"
                f"start={int(row['gene_start_bp'])}|"
                f"end={int(row['gene_end_bp'])}|"
                f"strand={int(row['strand'])}|"
                f"orientation={row['sequence_orientation']}"
            )

            sequence = str(row["genomic_sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")


whole_gene_genomic_fasta_path = (
    category_d_output_dir
    / "t2d_whole_gene_genomic_sequences_primary_only_v1.fasta"
)

write_whole_gene_genomic_fasta(
    t2d_whole_gene_genomic_sequences_primary_only_v1,
    whole_gene_genomic_fasta_path
)

print("Saved:", whole_gene_genomic_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output\t2d_whole_gene_genomic_sequences_primary_only_v1.fasta


In [169]:
whole_gene_genomic_validation_df = pd.DataFrame(
    list(whole_gene_genomic_validation.items()),
    columns=["metric", "value"]
)

validation_csv_path = (
    category_d_output_dir
    / "t2d_whole_gene_genomic_sequence_validation_primary_only_v1.csv"
)

whole_gene_genomic_validation_df.to_csv(
    validation_csv_path,
    index=False
)

whole_gene_genomic_validation_df

,metric,value
0,n_total_rows,1820
1,n_positive_rows,910
2,n_negative_rows,910
3,n_unique_gene_ids,1820
4,n_missing_genomic_sequences,0
5,n_length_mismatch_rows,0
6,n_invalid_sequence_character_rows,0
7,n_forward_oriented_sequences,924
8,n_reverse_complement_sequences,896


In [171]:
from pathlib import Path
import pandas as pd

# Dùng coordinate master đã tạo ở Category C
ensg_list_for_tss = (
    gene_coordinate_master_v1["ensembl_gene_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
)

print("Number of ENSG IDs:", len(ensg_list_for_tss))
ensg_list_for_tss.head()

Number of ENSG IDs: 1820


694     ENSG00000001626
1455    ENSG00000001629
543     ENSG00000002726
666     ENSG00000002822
1454    ENSG00000002919
Name: ensembl_gene_id, dtype: object

In [172]:
tss_work_dir = Path(
    r"D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss"
)
tss_work_dir.mkdir(parents=True, exist_ok=True)

ensg_txt_path = tss_work_dir / "t2d_1820_gene_ids_for_representative_tss.txt"

ensg_list_for_tss.to_csv(
    ensg_txt_path,
    index=False,
    header=False
)

print("Saved:", ensg_txt_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss\t2d_1820_gene_ids_for_representative_tss.txt


In [183]:
mane_file = tss_work_dir / "biomart_t2d_1820_mane_select_transcripts.tsv"
canonical_file = tss_work_dir / "biomart_t2d_1820_ensembl_canonical_transcripts.tsv"
all_transcripts_file = tss_work_dir / "biomart_t2d_1820_all_transcripts_for_tss_fallback.tsv"

df_mane_raw = pd.read_csv(mane_file, sep="\t", dtype=str)
df_canonical_raw = pd.read_csv(canonical_file, sep="\t", dtype=str)
df_all_transcripts_raw = pd.read_csv(all_transcripts_file, sep="\t", dtype=str)

print("MANE shape:", df_mane_raw.shape)
print("Canonical shape:", df_canonical_raw.shape)
print("All transcripts shape:", df_all_transcripts_raw.shape)

print("\nMANE columns:")
print(df_mane_raw.columns.tolist())

MANE shape: (1811, 8)
Canonical shape: (1820, 8)
All transcripts shape: (30935, 8)

MANE columns:
['Gene stable ID', 'Transcript stable ID', 'Gene name', 'Gene type', 'Chromosome/scaffold name', 'Strand', 'Transcript start (bp)', 'Transcript end (bp)']


In [184]:
df_mane_raw.columns.tolist()

['Gene stable ID',
 'Transcript stable ID',
 'Gene name',
 'Gene type',
 'Chromosome/scaffold name',
 'Strand',
 'Transcript start (bp)',
 'Transcript end (bp)']

In [185]:
df_canonical_raw.columns.tolist()

['Gene stable ID',
 'Transcript stable ID',
 'Gene name',
 'Gene type',
 'Chromosome/scaffold name',
 'Strand',
 'Transcript start (bp)',
 'Transcript end (bp)']

In [186]:
df_mane_raw.head()

,Gene stable ID,Transcript stable ID,Gene name,Gene type,Chromosome/scaffold name,Strand,Transcript start (bp),Transcript end (bp)
0,ENSG00000001626,ENST00000003084,CFTR,protein_coding,7,1,117480025,117668665
1,ENSG00000001629,ENST00000265742,ANKIB1,protein_coding,7,1,92245974,92401383
2,ENSG00000002726,ENST00000360937,AOC1,protein_coding,7,1,150852517,150861504
3,ENSG00000002822,ENST00000265854,MAD1L1,protein_coding,7,-1,1815795,2232945
4,ENSG00000002919,ENST00000359238,SNX11,protein_coding,17,1,48107766,48123601


In [187]:
df_canonical_raw.head()

,Gene stable ID,Transcript stable ID,Gene name,Gene type,Chromosome/scaffold name,Strand,Transcript start (bp),Transcript end (bp)
0,ENSG00000001626,ENST00000003084,CFTR,protein_coding,7,1,117480025,117668665
1,ENSG00000001629,ENST00000265742,ANKIB1,protein_coding,7,1,92245974,92401383
2,ENSG00000002726,ENST00000360937,AOC1,protein_coding,7,1,150852517,150861504
3,ENSG00000002822,ENST00000265854,MAD1L1,protein_coding,7,-1,1815795,2232945
4,ENSG00000002919,ENST00000359238,SNX11,protein_coding,17,1,48107766,48123601


In [188]:
print("Canonical shape:", df_canonical_raw.shape)
print(df_canonical_raw.columns.tolist())
df_canonical_raw.head()

Canonical shape: (1820, 8)
['Gene stable ID', 'Transcript stable ID', 'Gene name', 'Gene type', 'Chromosome/scaffold name', 'Strand', 'Transcript start (bp)', 'Transcript end (bp)']


,Gene stable ID,Transcript stable ID,Gene name,Gene type,Chromosome/scaffold name,Strand,Transcript start (bp),Transcript end (bp)
0,ENSG00000001626,ENST00000003084,CFTR,protein_coding,7,1,117480025,117668665
1,ENSG00000001629,ENST00000265742,ANKIB1,protein_coding,7,1,92245974,92401383
2,ENSG00000002726,ENST00000360937,AOC1,protein_coding,7,1,150852517,150861504
3,ENSG00000002822,ENST00000265854,MAD1L1,protein_coding,7,-1,1815795,2232945
4,ENSG00000002919,ENST00000359238,SNX11,protein_coding,17,1,48107766,48123601


In [189]:
tss_column_rename = {
    "Gene stable ID": "ensembl_gene_id",
    "Transcript stable ID": "representative_transcript_id",
    "Gene name": "transcript_gene_name",
    "Gene type": "transcript_gene_type",
    "Chromosome/scaffold name": "transcript_chromosome",
    "Strand": "transcript_strand",
    "Transcript start (bp)": "transcript_start_bp",
    "Transcript end (bp)": "transcript_end_bp"
}

df_mane = df_mane_raw.rename(columns=tss_column_rename).copy()
df_canonical = df_canonical_raw.rename(columns=tss_column_rename).copy()

In [190]:
numeric_tss_cols = [
    "transcript_strand",
    "transcript_start_bp",
    "transcript_end_bp"
]

for col in numeric_tss_cols:
    df_mane[col] = pd.to_numeric(df_mane[col], errors="coerce")
    df_canonical[col] = pd.to_numeric(df_canonical[col], errors="coerce")

In [191]:
tss_source_coverage = {
    "n_genes_final_dataset": gene_coordinate_master_v1["ensembl_gene_id"].nunique(),
    "n_mane_rows": len(df_mane),
    "n_unique_mane_genes": df_mane["ensembl_gene_id"].nunique(),
    "n_canonical_rows": len(df_canonical),
    "n_unique_canonical_genes": df_canonical["ensembl_gene_id"].nunique()
}

tss_source_coverage

{'n_genes_final_dataset': 1820,
 'n_mane_rows': 1811,
 'n_unique_mane_genes': 1811,
 'n_canonical_rows': 1820,
 'n_unique_canonical_genes': 1820}

In [192]:
mane_duplicate_genes = (
    df_mane["ensembl_gene_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

canonical_duplicate_genes = (
    df_canonical["ensembl_gene_id"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print("MANE genes with >1 row:", len(mane_duplicate_genes))
print("Canonical genes with >1 row:", len(canonical_duplicate_genes))

MANE genes with >1 row: 0
Canonical genes with >1 row: 0


In [193]:
df_mane["representative_transcript_rule"] = "MANE_Select"
df_mane["representative_transcript_priority"] = 1

df_canonical["representative_transcript_rule"] = "Ensembl_Canonical_Fallback"
df_canonical["representative_transcript_priority"] = 2

In [194]:
mane_gene_ids = set(df_mane["ensembl_gene_id"])
canonical_gene_ids = set(df_canonical["ensembl_gene_id"])

genes_without_mane = sorted(canonical_gene_ids - mane_gene_ids)

print("Genes without MANE Select:", len(genes_without_mane))
genes_without_mane

Genes without MANE Select: 9


['ENSG00000175164',
 'ENSG00000198695',
 'ENSG00000198763',
 'ENSG00000198786',
 'ENSG00000198840',
 'ENSG00000198886',
 'ENSG00000198888',
 'ENSG00000212907',
 'ENSG00000219481']

In [195]:
df_canonical_fallback = df_canonical[
    df_canonical["ensembl_gene_id"].isin(genes_without_mane)
].copy()

print("Canonical fallback shape:", df_canonical_fallback.shape)

Canonical fallback shape: (9, 10)


In [196]:
representative_transcripts_v1 = pd.concat(
    [
        df_mane,
        df_canonical_fallback
    ],
    ignore_index=True
)

representative_transcripts_v1 = (
    representative_transcripts_v1
    .sort_values(
        ["ensembl_gene_id", "representative_transcript_priority"]
    )
    .reset_index(drop=True)
)

print("Representative transcript table shape:",
      representative_transcripts_v1.shape)

print("Unique ENSG:",
      representative_transcripts_v1["ensembl_gene_id"].nunique())

representative_transcripts_v1[
    "representative_transcript_rule"
].value_counts()

Representative transcript table shape: (1820, 10)
Unique ENSG: 1820


representative_transcript_rule
MANE_Select                   1811
Ensembl_Canonical_Fallback       9
Name: count, dtype: int64

In [197]:
representative_transcripts_v1["representative_tss_bp"] = np.where(
    representative_transcripts_v1["transcript_strand"] == 1,
    representative_transcripts_v1["transcript_start_bp"],
    representative_transcripts_v1["transcript_end_bp"]
)

In [198]:
representative_transcripts_v1["representative_tss_rule"] = np.where(
    representative_transcripts_v1["transcript_strand"] == 1,
    "TSS_equals_transcript_start_for_positive_strand",
    "TSS_equals_transcript_end_for_negative_strand"
)

In [203]:
tss_logic_validation = {
    "n_rows": len(representative_transcripts_v1),
    "n_unique_genes": representative_transcripts_v1["ensembl_gene_id"].nunique(),
    "n_missing_transcript_id": int(
        representative_transcripts_v1["representative_transcript_id"].isna().sum()
    ),
    "n_missing_transcript_start": int(
        representative_transcripts_v1["transcript_start_bp"].isna().sum()
    ),
    "n_missing_transcript_end": int(
        representative_transcripts_v1["transcript_end_bp"].isna().sum()
    ),
    "n_missing_tss": int(
        representative_transcripts_v1["representative_tss_bp"].isna().sum()
    ),
    "n_invalid_start_gt_end": int(
        (
            representative_transcripts_v1["transcript_start_bp"]
            > representative_transcripts_v1["transcript_end_bp"]
        ).sum()
    ),
    "n_invalid_strand": int(
        (~representative_transcripts_v1["transcript_strand"].isin([1, -1])).sum()
    )
}

tss_logic_validation


{'n_rows': 1820,
 'n_unique_genes': 1820,
 'n_missing_transcript_id': 0,
 'n_missing_transcript_start': 0,
 'n_missing_transcript_end': 0,
 'n_missing_tss': 0,
 'n_invalid_start_gt_end': 0,
 'n_invalid_strand': 0}

In [204]:
representative_transcripts_v1[
    representative_transcripts_v1["representative_transcript_rule"]
    == "Ensembl_Canonical_Fallback"
][
    [
        "ensembl_gene_id",
        "transcript_gene_name",
        "representative_transcript_id",
        "transcript_chromosome",
        "transcript_strand",
        "transcript_start_bp",
        "transcript_end_bp",
        "representative_tss_bp"
    ]
]

,ensembl_gene_id,transcript_gene_name,representative_transcript_id,transcript_chromosome,transcript_strand,transcript_start_bp,transcript_end_bp,representative_tss_bp
1340,ENSG00000175164,ABO,ENST00000611156,9,-1,133255176,133275214,133275214
1635,ENSG00000198695,MT-ND6,ENST00000361681,MT,-1,14149,14673,14673
1638,ENSG00000198763,MT-ND2,ENST00000361453,MT,1,4470,5511,4470
1640,ENSG00000198786,MT-ND5,ENST00000361567,MT,1,12337,14148,12337
1645,ENSG00000198840,MT-ND3,ENST00000361227,MT,1,10059,10404,10059
1649,ENSG00000198886,MT-ND4,ENST00000361381,MT,1,10760,12137,10760
1650,ENSG00000198888,MT-ND1,ENST00000361390,MT,1,3307,4262,3307
1693,ENSG00000212907,MT-ND4L,ENST00000361335,MT,1,10470,10766,10470
1722,ENSG00000219481,NBPF1,ENST00000430580,1,-1,16562319,16613487,16613487


In [206]:
tss_cols_to_merge = [
    "ensembl_gene_id",
    "representative_transcript_id",
    "representative_transcript_rule",
    "representative_transcript_priority",
    "transcript_gene_name",
    "transcript_gene_type",
    "transcript_chromosome",
    "transcript_strand",
    "transcript_start_bp",
    "transcript_end_bp",
    "representative_tss_bp",
    "representative_tss_rule"
]

t2d_representative_tss_master_primary_only_v1 = (
    gene_coordinate_master_v1
    .merge(
        representative_transcripts_v1[tss_cols_to_merge],
        on="ensembl_gene_id",
        how="left"
    )
)

In [207]:
print(
    "Representative TSS Master shape:",
    t2d_representative_tss_master_primary_only_v1.shape
)

Representative TSS Master shape: (1820, 29)


In [208]:
t2d_representative_tss_master_primary_only_v1[
    "chromosome_match_gene_vs_transcript"
] = (
    t2d_representative_tss_master_primary_only_v1[
        "chromosome_or_scaffold"
    ].astype(str)
    ==
    t2d_representative_tss_master_primary_only_v1[
        "transcript_chromosome"
    ].astype(str)
)

t2d_representative_tss_master_primary_only_v1[
    "strand_match_gene_vs_transcript"
] = (
    t2d_representative_tss_master_primary_only_v1["strand"].astype(int)
    ==
    t2d_representative_tss_master_primary_only_v1[
        "transcript_strand"
    ].astype(int)
)

In [209]:
tss_master_validation = {
    "n_rows": len(t2d_representative_tss_master_primary_only_v1),
    "n_unique_gene_ids": (
        t2d_representative_tss_master_primary_only_v1["gene_id"].nunique()
    ),
    "n_positive_rows": int(
        (t2d_representative_tss_master_primary_only_v1["label"] == 1).sum()
    ),
    "n_negative_rows": int(
        (t2d_representative_tss_master_primary_only_v1["label"] == 0).sum()
    ),
    "n_missing_representative_transcripts": int(
        t2d_representative_tss_master_primary_only_v1[
            "representative_transcript_id"
        ].isna().sum()
    ),
    "n_missing_representative_tss": int(
        t2d_representative_tss_master_primary_only_v1[
            "representative_tss_bp"
        ].isna().sum()
    ),
    "n_mane_select": int(
        (
            t2d_representative_tss_master_primary_only_v1[
                "representative_transcript_rule"
            ] == "MANE_Select"
        ).sum()
    ),
    "n_ensembl_canonical_fallback": int(
        (
            t2d_representative_tss_master_primary_only_v1[
                "representative_transcript_rule"
            ] == "Ensembl_Canonical_Fallback"
        ).sum()
    ),
    "n_chromosome_mismatch": int(
        (~t2d_representative_tss_master_primary_only_v1[
            "chromosome_match_gene_vs_transcript"
        ]).sum()
    ),
    "n_strand_mismatch": int(
        (~t2d_representative_tss_master_primary_only_v1[
            "strand_match_gene_vs_transcript"
        ]).sum()
    )
}

tss_master_validation

{'n_rows': 1820,
 'n_unique_gene_ids': 1820,
 'n_positive_rows': 910,
 'n_negative_rows': 910,
 'n_missing_representative_transcripts': 0,
 'n_missing_representative_tss': 0,
 'n_mane_select': 1811,
 'n_ensembl_canonical_fallback': 9,
 'n_chromosome_mismatch': 0,
 'n_strand_mismatch': 0}

In [210]:
t2d_representative_tss_master_primary_only_v1[
    "representative_tss_master_version"
] = "Representative_TSS_Master_PrimaryOnly_v1"

t2d_representative_tss_master_primary_only_v1[
    "representative_transcript_selection_strategy"
] = "MANE_Select_preferred_else_Ensembl_Canonical"

In [211]:
representative_tss_master_output_path = (
    tss_work_dir
    / "t2d_representative_tss_master_primary_only_v1.csv"
)

t2d_representative_tss_master_primary_only_v1.to_csv(
    representative_tss_master_output_path,
    index=False
)

print("Saved:", representative_tss_master_output_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss\t2d_representative_tss_master_primary_only_v1.csv


In [212]:
canonical_fallback_audit_v1 = (
    representative_transcripts_v1[
        representative_transcripts_v1["representative_transcript_rule"]
        == "Ensembl_Canonical_Fallback"
    ]
    .copy()
)

canonical_fallback_audit_output_path = (
    tss_work_dir
    / "t2d_representative_tss_canonical_fallback_audit_v1.csv"
)

canonical_fallback_audit_v1.to_csv(
    canonical_fallback_audit_output_path,
    index=False
)

print("Saved:", canonical_fallback_audit_output_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss\t2d_representative_tss_canonical_fallback_audit_v1.csv


In [213]:
mt_genes_tss_master = (
    t2d_representative_tss_master_primary_only_v1[
        t2d_representative_tss_master_primary_only_v1[
            "transcript_chromosome"
        ].astype(str) == "MT"
    ]
    .copy()
)

print("MT genes shape:", mt_genes_tss_master.shape)

mt_genes_tss_master[
    [
        "gene_id",
        "gene_symbol",
        "label",
        "dataset_role",
        "representative_transcript_id",
        "representative_transcript_rule",
        "representative_tss_bp",
        "transcript_strand"
    ]
]

MT genes shape: (7, 33)


,gene_id,gene_symbol,label,dataset_role,representative_transcript_id,representative_transcript_rule,representative_tss_bp,transcript_strand
853,ENSG00000198886,MT-ND4,1,positive,ENST00000361381,Ensembl_Canonical_Fallback,10760,1
1024,ENSG00000198786,MT-ND5,1,positive,ENST00000361567,Ensembl_Canonical_Fallback,12337,1
1073,ENSG00000198695,MT-ND6,1,positive,ENST00000361681,Ensembl_Canonical_Fallback,14673,-1
1357,ENSG00000198840,MT-ND3,1,positive,ENST00000361227,Ensembl_Canonical_Fallback,10059,1
1396,ENSG00000198888,MT-ND1,1,positive,ENST00000361390,Ensembl_Canonical_Fallback,3307,1
1470,ENSG00000198763,MT-ND2,1,positive,ENST00000361453,Ensembl_Canonical_Fallback,4470,1
1684,ENSG00000212907,MT-ND4L,1,positive,ENST00000361335,Ensembl_Canonical_Fallback,10470,1


In [214]:
mt_genes_tss_master["label"].value_counts()

label
1    7
Name: count, dtype: int64

In [215]:
import pandas as pd
import numpy as np
from pathlib import Path

primary_nuclear_chromosomes = set([str(i) for i in range(1, 23)] + ["X", "Y"])

tss_master = t2d_representative_tss_master_primary_only_v1.copy()

# 1. Split nuclear vs MT
tss_master["is_nuclear_chromosome"] = (
    tss_master["transcript_chromosome"].astype(str).isin(primary_nuclear_chromosomes)
)

nuclear_tss_master_v1 = tss_master[
    tss_master["is_nuclear_chromosome"]
].copy()

mt_tss_audit_v1 = tss_master[
    ~tss_master["is_nuclear_chromosome"]
].copy()

print("Nuclear TSS master shape:", nuclear_tss_master_v1.shape)
print("MT audit shape:", mt_tss_audit_v1.shape)

print("\nNuclear label counts:")
print(nuclear_tss_master_v1["label"].value_counts())

print("\nMT label counts:")
print(mt_tss_audit_v1["label"].value_counts())

Nuclear TSS master shape: (1813, 34)
MT audit shape: (7, 34)

Nuclear label counts:
label
0    910
1    903
Name: count, dtype: int64

MT label counts:
label
1    7
Name: count, dtype: int64


In [216]:
positive_nuclear_tss = nuclear_tss_master_v1[
    nuclear_tss_master_v1["label"] == 1
].copy()

negative_nuclear_tss = nuclear_tss_master_v1[
    nuclear_tss_master_v1["label"] == 0
].copy()

n_positive_nuclear = len(positive_nuclear_tss)

negative_nuclear_sampled_tss = (
    negative_nuclear_tss
    .sample(n=n_positive_nuclear, random_state=42)
    .copy()
)

t2d_representative_tss_master_nuclear_balanced_v1 = pd.concat(
    [
        positive_nuclear_tss,
        negative_nuclear_sampled_tss
    ],
    ignore_index=True
)

t2d_representative_tss_master_nuclear_balanced_v1 = (
    t2d_representative_tss_master_nuclear_balanced_v1
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Balanced nuclear TSS master shape:",
      t2d_representative_tss_master_nuclear_balanced_v1.shape)

print(
    t2d_representative_tss_master_nuclear_balanced_v1["label"]
    .value_counts()
)

Balanced nuclear TSS master shape: (1806, 34)
label
0    903
1    903
Name: count, dtype: int64


In [217]:
nuclear_balanced_tss_validation = {
    "n_rows": len(t2d_representative_tss_master_nuclear_balanced_v1),
    "n_unique_gene_ids": (
        t2d_representative_tss_master_nuclear_balanced_v1["gene_id"].nunique()
    ),
    "n_positive_rows": int(
        (t2d_representative_tss_master_nuclear_balanced_v1["label"] == 1).sum()
    ),
    "n_negative_rows": int(
        (t2d_representative_tss_master_nuclear_balanced_v1["label"] == 0).sum()
    ),
    "n_mt_rows": int(
        (
            t2d_representative_tss_master_nuclear_balanced_v1[
                "transcript_chromosome"
            ].astype(str) == "MT"
        ).sum()
    ),
    "n_missing_tss": int(
        t2d_representative_tss_master_nuclear_balanced_v1[
            "representative_tss_bp"
        ].isna().sum()
    ),
    "n_missing_transcript": int(
        t2d_representative_tss_master_nuclear_balanced_v1[
            "representative_transcript_id"
        ].isna().sum()
    ),
    "n_duplicate_gene_ids": int(
        t2d_representative_tss_master_nuclear_balanced_v1[
            "gene_id"
        ].duplicated().sum()
    )
}

nuclear_balanced_tss_validation

{'n_rows': 1806,
 'n_unique_gene_ids': 1806,
 'n_positive_rows': 903,
 'n_negative_rows': 903,
 'n_mt_rows': 0,
 'n_missing_tss': 0,
 'n_missing_transcript': 0,
 'n_duplicate_gene_ids': 0}

In [218]:
tss_work_dir = Path(
    r"D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss"
)
tss_work_dir.mkdir(parents=True, exist_ok=True)

t2d_representative_tss_master_nuclear_balanced_v1[
    "tss_dataset_version"
] = "Representative_TSS_Master_NuclearBalanced_v1"

t2d_representative_tss_master_nuclear_balanced_v1[
    "mt_handling_rule"
] = (
    "Mitochondrial genes excluded from promoter/TSS-proximal genomic branch v1; "
    "negative class downsampled to preserve 1:1 balance."
)

nuclear_tss_output_path = (
    tss_work_dir / "t2d_representative_tss_master_nuclear_balanced_v1.csv"
)

mt_audit_output_path = (
    tss_work_dir / "t2d_mitochondrial_genes_excluded_from_promoter_branch_v1.csv"
)

t2d_representative_tss_master_nuclear_balanced_v1.to_csv(
    nuclear_tss_output_path,
    index=False
)

mt_tss_audit_v1.to_csv(
    mt_audit_output_path,
    index=False
)

print("Saved:", nuclear_tss_output_path)
print("Saved:", mt_audit_output_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss\t2d_representative_tss_master_nuclear_balanced_v1.csv
Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_tss\t2d_mitochondrial_genes_excluded_from_promoter_branch_v1.csv


In [219]:
DNA_COMPLEMENT_TABLE = str.maketrans({
    "A": "T",
    "T": "A",
    "C": "G",
    "G": "C",
    "N": "N"
})

def reverse_complement_dna(sequence: str) -> str:
    return sequence.translate(DNA_COMPLEMENT_TABLE)[::-1]

In [220]:
def extract_tss_proximal_sequence(
    chromosome: str,
    tss_bp: int,
    strand: int,
    genome_fasta,
    upstream_bp: int = 2000,
    downstream_bp: int = 500
) -> dict:
    """
    Extract fixed-length TSS-proximal sequence.

    Output length = upstream_bp + downstream_bp = 2500 bp.

    For + strand:
        [TSS - upstream_bp, TSS + downstream_bp - 1]

    For - strand:
        reference window = [TSS - downstream_bp + 1, TSS + upstream_bp]
        then reverse complement.
    """

    chromosome = str(chromosome)
    tss_bp = int(tss_bp)
    strand = int(strand)

    chrom_len = len(genome_fasta[chromosome])
    target_length = upstream_bp + downstream_bp

    if strand == 1:
        window_start_1based = tss_bp - upstream_bp
        window_end_1based = tss_bp + downstream_bp - 1
        sequence_orientation = "forward_gene_orientation"
    elif strand == -1:
        window_start_1based = tss_bp - downstream_bp + 1
        window_end_1based = tss_bp + upstream_bp
        sequence_orientation = "reverse_complement_gene_orientation"
    else:
        raise ValueError(f"Unexpected strand value: {strand}")

    # Boundary handling
    left_pad = max(0, 1 - window_start_1based)
    right_pad = max(0, window_end_1based - chrom_len)

    clipped_start_1based = max(1, window_start_1based)
    clipped_end_1based = min(chrom_len, window_end_1based)

    # Convert to pyfaidx slicing: 0-based start, end-exclusive
    fasta_start = clipped_start_1based - 1
    fasta_end = clipped_end_1based

    reference_sequence = str(
        genome_fasta[chromosome][fasta_start:fasta_end]
    ).upper()

    # Pad with N if window crosses chromosome boundary
    reference_sequence = (
        ("N" * left_pad)
        + reference_sequence
        + ("N" * right_pad)
    )

    if strand == 1:
        oriented_sequence = reference_sequence
    else:
        oriented_sequence = reverse_complement_dna(reference_sequence)

    return {
        "tss_window_start_1based": window_start_1based,
        "tss_window_end_1based": window_end_1based,
        "tss_window_clipped_start_1based": clipped_start_1based,
        "tss_window_clipped_end_1based": clipped_end_1based,
        "tss_upstream_bp": upstream_bp,
        "tss_downstream_bp": downstream_bp,
        "tss_proximal_sequence": oriented_sequence,
        "tss_proximal_sequence_length": len(oriented_sequence),
        "target_tss_window_length": target_length,
        "tss_window_length_match_check": len(oriented_sequence) == target_length,
        "tss_sequence_orientation": sequence_orientation,
        "left_boundary_N_padding": left_pad,
        "right_boundary_N_padding": right_pad,
        "total_N_padding": left_pad + right_pad
    }

In [221]:
tss_window_records = []

for _, row in t2d_representative_tss_master_nuclear_balanced_v1.iterrows():
    extracted = extract_tss_proximal_sequence(
        chromosome=row["transcript_chromosome"],
        tss_bp=row["representative_tss_bp"],
        strand=row["transcript_strand"],
        genome_fasta=genome,
        upstream_bp=2000,
        downstream_bp=500
    )

    tss_window_records.append(extracted)

df_tss_windows = pd.DataFrame(tss_window_records)

print("TSS windows shape:", df_tss_windows.shape)
df_tss_windows.head()

TSS windows shape: (1806, 14)


,tss_window_start_1based,tss_window_end_1based,tss_window_clipped_start_1based,tss_window_clipped_end_1based,tss_upstream_bp,tss_downstream_bp,tss_proximal_sequence,tss_proximal_sequence_length,target_tss_window_length,tss_window_length_match_check,tss_sequence_orientation,left_boundary_N_padding,right_boundary_N_padding,total_N_padding
0,36185497,36187996,36185497,36187996,2000,500,TCAACTTAGATGGGGAAACTGAGGCCCTGGGAACCACGGTGGCTCA...,2500,2500,True,forward_gene_orientation,0,0,0
1,44186689,44189188,44186689,44189188,2000,500,TGCAGCTCAAACTCCCCTAGCTTCCAGTTTGAGATGCTCTTCCCTT...,2500,2500,True,reverse_complement_gene_orientation,0,0,0
2,10939654,10942153,10939654,10942153,2000,500,CTGTCTCATTATTCTATTTTATTGGCTTGTCACTGTCTAAAAATTA...,2500,2500,True,reverse_complement_gene_orientation,0,0,0
3,2962907,2965406,2962907,2965406,2000,500,CGTGGCTAGCACAGGGACTTGAGGTCCGAGGCTCTGTCGCCAGCTC...,2500,2500,True,reverse_complement_gene_orientation,0,0,0
4,666132,668631,666132,668631,2000,500,GTTGGGTGTGTTGGGGATGGTGCCGTCCTGACCTGGCCCAGGTCCA...,2500,2500,True,forward_gene_orientation,0,0,0


In [222]:
t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1 = pd.concat(
    [
        t2d_representative_tss_master_nuclear_balanced_v1.reset_index(drop=True),
        df_tss_windows.reset_index(drop=True)
    ],
    axis=1
)

t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
    "regulatory_sequence_source"
] = "Ensembl Homo sapiens GRCh38 primary assembly FASTA"

t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
    "regulatory_sequence_scope"
] = "TSS-proximal window: 2000 bp upstream + 500 bp downstream"

t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
    "regulatory_sequence_version"
] = "TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1"

In [223]:
allowed_dna_characters = set("ACGTN")

invalid_tss_sequence_character_rows = (
    t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
        "tss_proximal_sequence"
    ]
    .apply(lambda seq: not set(str(seq)).issubset(allowed_dna_characters))
)

tss_proximal_regulatory_validation = {
    "n_rows": len(t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1),
    "n_unique_gene_ids": (
        t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
            "gene_id"
        ].nunique()
    ),
    "n_positive_rows": int(
        (
            t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
                "label"
            ] == 1
        ).sum()
    ),
    "n_negative_rows": int(
        (
            t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
                "label"
            ] == 0
        ).sum()
    ),
    "n_missing_sequences": int(
        t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
            "tss_proximal_sequence"
        ].isna().sum()
    ),
    "n_sequence_length_not_2500": int(
        (
            t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
                "tss_proximal_sequence_length"
            ] != 2500
        ).sum()
    ),
    "n_length_mismatch_rows": int(
        (
            ~t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
                "tss_window_length_match_check"
            ]
        ).sum()
    ),
    "n_invalid_sequence_character_rows": int(
        invalid_tss_sequence_character_rows.sum()
    ),
    "n_mt_rows": int(
        (
            t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
                "transcript_chromosome"
            ].astype(str) == "MT"
        ).sum()
    ),
    "n_rows_with_boundary_padding": int(
        (
            t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
                "total_N_padding"
            ] > 0
        ).sum()
    ),
    "max_total_N_padding": int(
        t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1[
            "total_N_padding"
        ].max()
    )
}

tss_proximal_regulatory_validation

{'n_rows': 1806,
 'n_unique_gene_ids': 1806,
 'n_positive_rows': 903,
 'n_negative_rows': 903,
 'n_missing_sequences': 0,
 'n_sequence_length_not_2500': 0,
 'n_length_mismatch_rows': 0,
 'n_invalid_sequence_character_rows': 0,
 'n_mt_rows': 0,
 'n_rows_with_boundary_padding': 0,
 'max_total_N_padding': 0}

In [224]:
category_d_output_dir = Path(
    r"D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output"
)
category_d_output_dir.mkdir(parents=True, exist_ok=True)

tss_regulatory_csv_path = (
    category_d_output_dir
    / "t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv"
)

t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1.to_csv(
    tss_regulatory_csv_path,
    index=False
)

print("Saved:", tss_regulatory_csv_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output\t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv


In [225]:
def write_tss_proximal_regulatory_fasta(df: pd.DataFrame, fasta_path: Path):
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            header = (
                f">{row['gene_id']}|"
                f"{row['gene_symbol']}|"
                f"{row['representative_transcript_id']}|"
                f"label={int(row['label'])}|"
                f"chr={row['transcript_chromosome']}|"
                f"tss={int(row['representative_tss_bp'])}|"
                f"strand={int(row['transcript_strand'])}|"
                f"window={int(row['tss_window_start_1based'])}-"
                f"{int(row['tss_window_end_1based'])}|"
                f"orientation={row['tss_sequence_orientation']}"
            )

            sequence = str(row["tss_proximal_sequence"])

            f.write(header + "\n")

            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")


tss_regulatory_fasta_path = (
    category_d_output_dir
    / "t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.fasta"
)

write_tss_proximal_regulatory_fasta(
    t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_v1,
    tss_regulatory_fasta_path
)

print("Saved:", tss_regulatory_fasta_path)

Saved: D:\khoane\MAHE\Project\dataset\UniPro\category_d_genomic_sequence_output\t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.fasta


In [226]:
tss_regulatory_validation_df = pd.DataFrame(
    list(tss_proximal_regulatory_validation.items()),
    columns=["metric", "value"]
)

tss_regulatory_validation_path = (
    category_d_output_dir
    / "t2d_tss_proximal_regulatory_validation_nuclear_balanced_v1.csv"
)

tss_regulatory_validation_df.to_csv(
    tss_regulatory_validation_path,
    index=False
)

tss_regulatory_validation_df

,metric,value
0,n_rows,1806
1,n_unique_gene_ids,1806
2,n_positive_rows,903
3,n_negative_rows,903
4,n_missing_sequences,0
5,n_sequence_length_not_2500,0
6,n_length_mismatch_rows,0
7,n_invalid_sequence_character_rows,0
8,n_mt_rows,0
9,n_rows_with_boundary_padding,0
